# Stochastic Modelling and Risk Assessment of Aeronautical Revenue Stability
## Main simulation notebook

Run the cells in order. `DATA.xlsx` must be in the same folder as this notebook. Random seed 42.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Load the consolidated file
df_flights = pd.read_excel("DATA.xlsx", sheet_name="Flight Records")
df_monthly = pd.read_excel("DATA.xlsx", sheet_name="Monthly Summary")
df_annual = pd.read_excel("DATA.xlsx", sheet_name="Annual Summary")

print(f"Flight Records: {len(df_flights)} rows")
print(f"Monthly Summary: {len(df_monthly)} rows")
print(f"Annual Summary: {len(df_annual)} rows")
print(f"\nFlight Records columns: {df_flights.columns.tolist()}")
print(f"\nAnnual Summary:")
print(df_annual.to_string(index=False))


In [ ]:
# ── CELL 2: Core data preparation ──

# Filter chargeable flights only (TOTAL > 0)
df_flights['TOTAL'] = pd.to_numeric(df_flights['TOTAL'], errors='coerce')
df_flights['AUW'] = pd.to_numeric(df_flights['AUW'], errors='coerce')
df_flights['YEAR'] = pd.to_numeric(df_flights['YEAR'], errors='coerce')
df_flights['MONTH'] = pd.to_numeric(df_flights['MONTH'], errors='coerce')
df_flights = df_flights[df_flights['TOTAL'] > 0].reset_index(drop=True)

# Annual revenue from flight records
annual_revenue = df_flights.groupby('YEAR')['TOTAL'].sum()

# Monthly revenue from monthly summary
df_monthly['REVENUE'] = pd.to_numeric(df_monthly['REVENUE'], errors='coerce')
df_monthly['YEAR'] = pd.to_numeric(df_monthly['YEAR'], errors='coerce')
df_monthly['MONTH'] = pd.to_numeric(df_monthly['MONTH'], errors='coerce')

# NBS Inflation rates 2008-2025
inflation_data = {
    2008: 0.1158, 2009: 0.1254, 2010: 0.1374,
    2011: 0.1083, 2012: 0.1222, 2013: 0.0850,
    2014: 0.0805, 2015: 0.0901, 2016: 0.1570,
    2017: 0.1650, 2018: 0.1210, 2019: 0.1140,
    2020: 0.1325, 2021: 0.1695, 2022: 0.1885,
    2023: 0.2466, 2024: 0.3324, 2025: 0.2333
}

inflation_df = pd.DataFrame(list(inflation_data.items()),
                             columns=['YEAR', 'INFLATION_RATE'])

# Cumulative inflation factor from 2008 up to PREVIOUS year (lagged)
def cumulative_factor_lagged(year):
    factor = 1.0
    for yr in range(2008, year):
        factor *= (1 + inflation_data[yr])
    return factor

# Simulation parameters
mean_inflation = inflation_df['INFLATION_RATE'].mean()

# Population standard deviation — consistent with the K-S test
std_inflation = inflation_df['INFLATION_RATE'].std(ddof=0)

mean_future_inflation = mean_inflation
std_future_inflation = std_inflation

print(f"Mean inflation: {mean_inflation*100:.2f}%")
print(f"Population SD:  {std_inflation*100:.2f}%")

print("✅ Core data ready")
print(f"Total chargeable flights: {len(df_flights)}")
print(f"Annual revenue summary:")
print(annual_revenue.to_string())
print(f"\nInflation parameters:")
print(f"Mean: {mean_inflation*100:.2f}%")
print(f"Std:  {std_inflation*100:.2f}%")
print(f"\nCumulative factor by 2025: {cumulative_factor_lagged(2025):.4f}")


In [ ]:
# ── CELL 3: Distribution Tests (K-S) ──

# 1. Monthly Revenue
monthly_revenue = df_monthly['REVENUE'].values
mean_rev = np.mean(monthly_revenue)
std_rev = np.std(monthly_revenue)
ks_stat_rev, ks_p_rev = stats.kstest(monthly_revenue, 'norm', args=(mean_rev, std_rev))

# 2. Inflation Rates
inflation_rates = inflation_df['INFLATION_RATE'].values
mean_inf = np.mean(inflation_rates)
std_inf = np.std(inflation_rates)
ks_stat_inf, ks_p_inf = stats.kstest(inflation_rates, 'norm', args=(mean_inf, std_inf))

# 3. Monthly Chargeable Movements
monthly_movements = df_monthly['FLIGHTS'].values
mean_mov = np.mean(monthly_movements)
std_mov = np.std(monthly_movements)
ks_stat_mov, ks_p_mov = stats.kstest(monthly_movements, 'norm', args=(mean_mov, std_mov))

print("KOLMOGOROV-SMIRNOV DISTRIBUTION TESTS")
print("=" * 70)
print(f"{'Variable':<30} {'Mean':<15} {'Std':<12} {'K-S Stat':<12} {'P-value':<10} {'Result'}")
print("-" * 70)

tests = [
    ("Monthly Revenue (₦)", f"₦{mean_rev:,.0f}", f"₦{std_rev:,.0f}", 
     ks_stat_rev, ks_p_rev),
    ("Inflation Rates", f"{mean_inf*100:.2f}%", f"{std_inf*100:.2f}%", 
     ks_stat_inf, ks_p_inf),
    ("Chargeable Movements", f"{mean_mov:.1f}", f"{std_mov:.1f}", 
     ks_stat_mov, ks_p_mov),
]

for name, mean, std, stat, p in tests:
    result = "Normal ✅" if p > 0.05 else "Not Normal ❌"
    print(f"{name:<30} {mean:<15} {std:<12} {stat:<12.4f} {p:<10.4f} {result}")


In [ ]:
# ── CELL 4: Real Value Erosion (2008-2025) ──

cumulative_index = 1.0
real_value = [1.0]

for year in range(2009, 2026):
    rate = inflation_data[year]
    cumulative_index = cumulative_index * (1 + rate)
    real_value.append(1 / cumulative_index)

years = list(range(2008, 2026))

# Print table
print(f"{'Year':<8} {'Cumulative Inflation':<25} {'Real Value of ₦1 (2008)'}")
print("-" * 55)
for i, year in enumerate(years):
    print(f"{year:<8} {(1/real_value[i] - 1)*100:<25.1f}% ₦{real_value[i]:.4f}")

print(f"\nTotal purchasing power lost: {((1 - real_value[-1]) * 100):.1f}%")
print(f"By 2025, ₦1 of 2008 revenue is worth only ₦{real_value[-1]:.4f}")

# Plot
plt.figure(figsize=(12, 5))
plt.plot(years, real_value, color='red', marker='o', linewidth=2)
plt.fill_between(years, real_value, 1, alpha=0.1, color='red')
plt.title('Real Value of ₦1 of 2008 NAMA Yola Revenue (2008–2025)', fontsize=13)
plt.xlabel('Year')
plt.ylabel('Real Value (₦, 2008 base)')
plt.xticks(years, rotation=45)
plt.axhline(y=real_value[-1], color='gray', linestyle='--', linewidth=1)
plt.annotate('₦0.09 remaining\n(90.9% lost)',
             xy=(2025, real_value[-1]),
             xytext=(2019, 0.25),
             arrowprops=dict(arrowstyle='->', color='black'),
             fontsize=10, color='red')
plt.tight_layout()
plt.show()


In [ ]:
# ── CELL 5: Retrospective Revenue Gap (2021-2025) ──

# Annual actual revenue
annual_actual = df_monthly.groupby('YEAR')['REVENUE'].sum().reset_index()
annual_actual.columns = ['YEAR', 'ACTUAL_REVENUE']

# Annualise 2025 (9 months only)
annual_actual.loc[annual_actual['YEAR'] == 2025, 'ACTUAL_REVENUE'] = \
    annual_actual.loc[annual_actual['YEAR'] == 2025, 'ACTUAL_REVENUE'] * (12/9)

# Anchor = 2021 actual revenue
anchor_2021 = annual_actual.loc[annual_actual['YEAR'] == 2021, 'ACTUAL_REVENUE'].values[0]

# Compute adjusted revenue using lagged cumulative inflation from 2008
annual_actual['INFLATION_FACTOR'] = annual_actual['YEAR'].apply(
    lambda yr: cumulative_factor_lagged(int(yr)))

annual_actual['ADJUSTED_REVENUE'] = anchor_2021 * annual_actual['INFLATION_FACTOR']
annual_actual['REVENUE_GAP'] = annual_actual['ADJUSTED_REVENUE'] - annual_actual['ACTUAL_REVENUE']
annual_actual['GAP_PERCENT'] = (annual_actual['REVENUE_GAP'] / annual_actual['ADJUSTED_REVENUE']) * 100

print("RETROSPECTIVE REVENUE GAP (2021-2025)")
print("=" * 80)
print(f"{'Year':<8} {'Actual (₦M)':<16} {'Adjusted (₦M)':<18} {'Gap (₦M)':<14} {'Gap %'}")
print("-" * 80)
for _, row in annual_actual.iterrows():
    print(f"{int(row['YEAR']):<8} ₦{row['ACTUAL_REVENUE']/1e6:<14.2f} "
          f"₦{row['ADJUSTED_REVENUE']/1e6:<16.2f} "
          f"₦{row['REVENUE_GAP']/1e6:<12.2f} {row['GAP_PERCENT']:.1f}%")

total_gap = annual_actual['REVENUE_GAP'].sum()
print(f"\nTotal revenue gap 2021-2025: ₦{total_gap/1e6:.1f} Million")


In [ ]:
# ── CELL 6: Retrospective Gap Chart ──

years_retro = annual_actual['YEAR'].astype(int).values
actual = annual_actual['ACTUAL_REVENUE'].values
adjusted = annual_actual['ADJUSTED_REVENUE'].values
gap = annual_actual['REVENUE_GAP'].values
gap_pct = annual_actual['GAP_PERCENT'].values

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Top chart: Actual vs Adjusted
ax1.bar(years_retro - 0.2, actual/1e6, width=0.4,
        label='Actual Revenue (Nominal)', color='steelblue')
ax1.bar(years_retro + 0.2, adjusted/1e6, width=0.4,
        label='Inflation-Adjusted Revenue (Should Have Earned)',
        color='red', alpha=0.7)
ax1.set_title('NAMA Yola: Actual vs Inflation-Adjusted Revenue (2021–2025)', fontsize=13)
ax1.set_xlabel('Year')
ax1.set_ylabel('Revenue (₦ Millions)')
ax1.legend()
ax1.set_xticks(years_retro)

# Bottom chart: Revenue Gap
ax2.bar(years_retro, gap/1e6, color='darkred', alpha=0.8)
ax2.set_title('Annual Revenue Gap — Purchasing Power Lost (2021–2025)', fontsize=13)
ax2.set_xlabel('Year')
ax2.set_ylabel('Gap (₦ Millions)')
ax2.set_xticks(years_retro)

# Add gap percentage labels
for yr, g, pct in zip(years_retro, gap, gap_pct):
    ax2.text(yr, g/1e6 + 1, f'{pct:.1f}%',
             ha='center', fontsize=10, color='darkred')

plt.tight_layout()
plt.show()

print(f"\nTotal revenue gap 2021-2025: ₦{gap.sum()/1e6:.1f} Million")


In [ ]:
# ── CELL 7: Monte Carlo Retrospective Validation (±5% uncertainty) ──

np.random.seed(42)
N_ITERATIONS = 10000

# ±5 percentage-point uncertainty around each historical inflation rate
std_uncertainty = 0.05

# IMPORTANT:
# Use the same 2021 baseline revenue anchor as the deterministic
# retrospective analysis so that both methods are directly comparable.
anchor_2021 = annual_actual.loc[
    annual_actual['YEAR'] == 2021,
    'ACTUAL_REVENUE'
].values[0]

mc_results_retro = {}

for year in [2021, 2022, 2023, 2024, 2025]:

    simulated_gaps = []
    simulated_real_revenues = []

    for i in range(N_ITERATIONS):

        # Build stochastic cumulative inflation factor
        # using historical rates from 2008 up to the
        # year preceding the target year.
        A_t = 1.0

        for yr in range(2008, year):

            actual_rate = inflation_data[yr]

            sampled_rate = np.random.normal(
                actual_rate,
                std_uncertainty
            )

            # Prevent unrealistic negative/extreme inflation draws
            sampled_rate = np.clip(
                sampled_rate,
                0.01,
                0.80
            )

            A_t *= (1 + sampled_rate)

        # Inflation-required revenue using the SAME
        # 2021 baseline anchor as the deterministic model
        R_r = anchor_2021 * A_t

        # Gap relative to the 2021 baseline anchor
        G = R_r - anchor_2021

        simulated_real_revenues.append(R_r)
        simulated_gaps.append(G)

    mc_results_retro[year] = {
        'anchor_revenue': anchor_2021,
        'mean_real_revenue': np.mean(simulated_real_revenues),
        'mean_gap': np.mean(simulated_gaps),
        'gap_5th': np.percentile(simulated_gaps, 5),
        'gap_95th': np.percentile(simulated_gaps, 95),

        # Gap as a percentage of the inflation-required benchmark
        'gap_percent': (
            np.mean(simulated_gaps)
            / np.mean(simulated_real_revenues)
        ) * 100
    }


# ── Validation table ──

print("MONTE CARLO RETROSPECTIVE VALIDATION (±5% uncertainty)")
print("=" * 90)

print(
    f"{'Year':<8}"
    f"{'Deterministic (₦M)':<22}"
    f"{'MC Mean (₦M)':<20}"
    f"{'Diff %':<10}"
    f"{'90% CI (₦M)'}"
)

print("-" * 90)

# Deterministic gaps from the same 2021-anchor model
det_gaps = {
    yr: annual_actual.loc[
        annual_actual['YEAR'] == yr,
        'REVENUE_GAP'
    ].values[0]
    for yr in [2021, 2022, 2023, 2024, 2025]
}

for year, res in mc_results_retro.items():

    det = det_gaps[year]
    mc = res['mean_gap']

    diff = ((mc - det) / det) * 100

    within = (
        "VALIDATED"
        if res['gap_5th'] <= det <= res['gap_95th']
        else "NOT VALIDATED"
    )

    print(
        f"{year:<8}"
        f"₦{det/1e6:<20.2f}"
        f"₦{mc/1e6:<18.2f}"
        f"{diff:+.1f}%"
        f"   ₦{res['gap_5th']/1e6:.1f}M — "
        f"₦{res['gap_95th']/1e6:.1f}M   "
        f"{within}"
    )


In [ ]:
# ── CELL 8: MC Retrospective Chart with Confidence Bands ──

years_retro = list(mc_results_retro.keys())
actual_revs = [annual_actual.loc[annual_actual['YEAR'] == yr, 'ACTUAL_REVENUE'].values[0] / 1e6 for yr in years_retro]
mean_real = [mc_results_retro[yr]['mean_real_revenue']/1e6 for yr in years_retro]
mean_gaps = [mc_results_retro[yr]['mean_gap']/1e6 for yr in years_retro]
gap_5th = [mc_results_retro[yr]['gap_5th']/1e6 for yr in years_retro]
gap_95th = [mc_results_retro[yr]['gap_95th']/1e6 for yr in years_retro]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Top chart
ax1.plot(years_retro, actual_revs, 'bo-', linewidth=2,
         markersize=8, label='Actual Nominal Revenue')
ax1.plot(years_retro, mean_real, 'ro-', linewidth=2,
         markersize=8, label='MC Mean Inflation-Required Revenue')
ax1.fill_between(years_retro,
                 [gap_5th[i] + actual_revs[i] for i in range(len(years_retro))],
                 [gap_95th[i] + actual_revs[i] for i in range(len(years_retro))],
                 alpha=0.2, color='red', label='90% Confidence Band')
ax1.set_title('Monte Carlo Retrospective: Actual vs Inflation-Required Revenue (2021–2025)',
              fontsize=12)
ax1.set_xlabel('Year')
ax1.set_ylabel('Revenue (₦ Millions)')
ax1.legend()
ax1.set_xticks(years_retro)

# Bottom chart
ax2.plot(years_retro, mean_gaps, 'ro-', linewidth=2,
         markersize=8, label='MC Mean Gap')
ax2.fill_between(years_retro, gap_5th, gap_95th,
                 alpha=0.2, color='red', label='90% Confidence Band')
ax2.plot(years_retro, gap_5th, 'r--', linewidth=1, alpha=0.5, label='5th Percentile')
ax2.plot(years_retro, gap_95th, 'r--', linewidth=1, alpha=0.5, label='95th Percentile')

# Add deterministic gap points
det_gap_values = [det_gaps[yr]/1e6 for yr in years_retro]
ax2.scatter(years_retro, det_gap_values, color='darkred', zorder=5,
            s=80, label='Deterministic Gap', marker='D')

for yr, gap in zip(years_retro, mean_gaps):
    ax2.annotate(f'₦{gap:.1f}M', xy=(yr, gap),
                 xytext=(0, 10), textcoords='offset points',
                 ha='center', fontsize=9, color='darkred')

ax2.set_title('MC Revenue Gap with 90% Confidence Bands (2021–2025)', fontsize=12)
ax2.set_xlabel('Year')
ax2.set_ylabel('Revenue Gap (₦ Millions)')
ax2.legend()
ax2.set_xticks(years_retro)

plt.tight_layout()
plt.show()

print(f"\nTotal MC gap 2021-2025: ₦{sum(mean_gaps):.1f} Million")
print(f"90% CI: ₦{sum(gap_5th):.1f}M — ₦{sum(gap_95th):.1f}M")


In [ ]:
# ── CELL 9: Sensitivity Analysis (g = 15% to 50%) — Corrected CAGR ──
np.random.seed(42)
N_ITERATIONS = 10000

# Base parameters
base_revenue = annual_revenue[2025] * (12/9)
base_movements = df_monthly[df_monthly['YEAR'] == 2025]['FLIGHTS'].sum() * (12/9)
avg_charge_per_flight = base_revenue / base_movements
A_base = cumulative_factor_lagged(2025)

# Corrected CAGR using actual flights only
actual_2021 = df_monthly[df_monthly['YEAR'] == 2021]['ACTUAL_FLIGHTS'].sum()
actual_2024 = df_monthly[df_monthly['YEAR'] == 2024]['ACTUAL_FLIGHTS'].sum()
cagr_movements = (actual_2024 / actual_2021) ** (1/3) - 1

print(f"Base annual revenue (2025 annualised): ₦{base_revenue/1e6:.2f}M")
print(f"Base annual movements: {base_movements:.0f}")
print(f"Avg charge per flight: ₦{avg_charge_per_flight:,.0f}")
print(f"Movement CAGR (actual flights only): {cagr_movements*100:.2f}%")
print(f"Accumulated inflation factor (A_base): {A_base:.4f}")

# Sensitivity parameters
g_values = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
proj_years = [2026, 2027, 2028, 2029, 2030]

# Store results
sensitivity_results = {}

for g in g_values:
    yearly_results = {}
    
    for year in proj_years:
        simulated_nominal = []
        simulated_required = []
        simulated_gaps = []
        gap_pcts = []
        
        for i in range(N_ITERATIONS):
            # Sample future inflation
            n_future_years = year - 2025
            future_inflations = np.random.normal(mean_future_inflation,
                                                  std_future_inflation,
                                                  n_future_years)
            future_inflations = np.clip(future_inflations, 0.01, 0.80)
            
            # Projected movements — flat trend (+0.46%/yr)
            years_from_base = year - 2025
            projected_movements = base_movements * ((1 + cagr_movements) ** years_from_base)
            
            # Adjusted rates grow by g annually
            adjusted_charge = avg_charge_per_flight * ((1 + g) ** years_from_base)
            nominal_revenue = projected_movements * adjusted_charge
            
            # Required revenue = full inflation indexing benchmark
            future_A_t = np.prod(1 + future_inflations)
            full_A_t = A_base * future_A_t
            adjusted_charge_s2 = avg_charge_per_flight * full_A_t
            R_r = projected_movements * adjusted_charge_s2
            
            # Gap
            G = R_r - nominal_revenue
            gap_pct = G / R_r

            simulated_nominal.append(nominal_revenue)
            simulated_required.append(R_r)
            simulated_gaps.append(G)
            gap_pcts.append(gap_pct)
        
        yearly_results[year] = {
            'mean_nominal': np.mean(simulated_nominal),
            'mean_required': np.mean(simulated_required),
            'mean_gap': np.mean(simulated_gaps),
            'gap_5th': np.percentile(simulated_gaps, 5),
            'gap_95th': np.percentile(simulated_gaps, 95),
            'gap_percent': np.mean(gap_pcts) * 100
        }
    
    sensitivity_results[g] = yearly_results
    print(f"g={g*100:.0f}% computed ✅")

print("\nSensitivity analysis complete")


In [ ]:
# ── CELL 10: Sensitivity Analysis Results Table & Chart ──

# Print results table
print("SENSITIVITY ANALYSIS RESULTS — Gap % by Adjustment Rate and Year")
print("=" * 75)
print(f"{'g':<8}", end="")
for year in proj_years:
    print(f"{year:<12}", end="")
print()
print("-" * 75)

for g in g_values:
    print(f"{g*100:.0f}%{'':<5}", end="")
    for year in proj_years:
        gap_pct = sensitivity_results[g][year]['gap_percent']
        print(f"{gap_pct:.1f}%{'':<6}", end="")
    print()

# Plot sensitivity curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(g_values)))

# Left chart: Gap % by year for each g
for i, g in enumerate(g_values):
    gap_pcts = [sensitivity_results[g][yr]['gap_percent'] for yr in proj_years]
    ax1.plot(proj_years, gap_pcts, marker='o', color=colors[i],
             linewidth=2, label=f'g={g*100:.0f}%')

ax1.set_title('Revenue Gap % by Year\nUnder Different Adjustment Rates', fontsize=12)
ax1.set_xlabel('Year')
ax1.set_ylabel('Gap as % of Inflation-Required Revenue')
ax1.legend(loc='upper left', fontsize=9)
ax1.set_xticks(proj_years)
ax1.grid(True, alpha=0.3)

# Right chart: Gap % by adjustment rate for each year
for year in proj_years:
    gap_pcts = [sensitivity_results[g][year]['gap_percent'] for g in g_values]
    ax2.plot([g*100 for g in g_values], gap_pcts, marker='o',
             linewidth=2, label=str(year))

ax2.set_title('Revenue Gap % by Adjustment Rate\nFor Each Projection Year', fontsize=12)
ax2.set_xlabel('Annual Tariff Adjustment Rate (%)')
ax2.set_ylabel('Gap as % of Inflation-Required Revenue')
ax2.legend(title='Year', fontsize=9)
ax2.set_xticks([g*100 for g in g_values])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# ── CELL 11: Baseline Scenario (S1) with MC Risk Metrics ──
np.random.seed(42)
N_ITERATIONS = 10000

# Sustainability threshold — real value preservation
# NAMA is currently collecting 10% of real value
# We define sustainability threshold as 20% of inflation-required revenue
# i.e. gap % must be below 80%
SUSTAINABILITY_THRESHOLD = 0.80

baseline_results = {}
baseline_all_gaps = {year: [] for year in proj_years}
baseline_all_nominal = {year: [] for year in proj_years}

for year in proj_years:
    simulated_nominal = []
    simulated_required = []
    simulated_gaps = []

    for i in range(N_ITERATIONS):
        n_future_years = year - 2025
        future_inflations = np.random.normal(mean_future_inflation,
                                              std_future_inflation,
                                              n_future_years)
        future_inflations = np.clip(future_inflations, 0.01, 0.80)

        # Baseline: rates frozen, movements flat (+0.46%)
        years_from_base = year - 2025
        projected_movements = base_movements * ((1 + cagr_movements) ** years_from_base)
        nominal_revenue = projected_movements * avg_charge_per_flight

        # Required revenue
        future_A_t = np.prod(1 + future_inflations)
        full_A_t = A_base * future_A_t
        R_r = projected_movements * (avg_charge_per_flight * full_A_t)

        G = R_r - nominal_revenue
        gap_pct = G / R_r

        simulated_nominal.append(nominal_revenue)
        simulated_required.append(R_r)
        simulated_gaps.append(G)
        baseline_all_gaps[year].append(gap_pct)
        baseline_all_nominal[year].append(nominal_revenue)

    gaps_array = np.array(simulated_gaps)
    gap_pcts_array = np.array(baseline_all_gaps[year])

    baseline_results[year] = {
        'mean_nominal': np.mean(simulated_nominal),
        'mean_required': np.mean(simulated_required),
        'mean_gap': np.mean(gaps_array),
        'std_gap': np.std(gaps_array),
        'gap_5th': np.percentile(gaps_array, 5),
        'gap_95th': np.percentile(gaps_array, 95),
        'gap_percent': np.mean(gap_pcts_array) * 100,
        'prob_below_threshold': np.mean(gap_pcts_array > SUSTAINABILITY_THRESHOLD) * 100,
        'prob_sustain': np.mean(gap_pcts_array <= SUSTAINABILITY_THRESHOLD) * 100,
    }

# Print results
print("SCENARIO 1 — BASELINE (Frozen Rates, Flat Traffic)")
print("=" * 90)
print(f"{'Year':<8} {'Nominal(₦M)':<14} {'Required(₦M)':<16} {'Gap(₦M)':<12} "
      f"{'Std(₦M)':<12} {'Gap%':<8} {'P(unsustainable)':<20} {'90% CI (₦M)'}")
print("-" * 90)

for year, res in baseline_results.items():
    print(f"{year:<8} ₦{res['mean_nominal']/1e6:<12.2f} ₦{res['mean_required']/1e6:<14.2f} "
          f"₦{res['mean_gap']/1e6:<10.2f} ₦{res['std_gap']/1e6:<10.2f} "
          f"{res['gap_percent']:<8.1f} {res['prob_below_threshold']:<20.1f}% "
          f"₦{res['gap_5th']/1e6:.1f}M–₦{res['gap_95th']/1e6:.1f}M")

total_gap_s1 = sum([baseline_results[yr]['mean_gap'] for yr in proj_years])
print(f"\nTotal projected gap 2026-2030 (Baseline): ₦{total_gap_s1/1e6:.1f} Million")
print(f"Probability of unsustainable revenue in 2030: {baseline_results[2030]['prob_below_threshold']:.1f}%")


In [ ]:
# ── CELL 12: Policy Scenarios (S2=15%, S3=30%, S4=50%) with MC Risk Metrics ──
np.random.seed(42)
N_ITERATIONS = 10000

g_scenarios = {
    'S2_Conservative': 0.15,
    'S3_Moderate': 0.30,
    'S4_Aggressive': 0.50
}

scenario_results = {}

for scenario_name, g in g_scenarios.items():
    yearly_results = {}

    for year in proj_years:
        simulated_nominal = []
        simulated_required = []
        simulated_gaps = []
        gap_pcts = []

        for i in range(N_ITERATIONS):
            n_future_years = year - 2025
            future_inflations = np.random.normal(mean_future_inflation,
                                                  std_future_inflation,
                                                  n_future_years)
            future_inflations = np.clip(future_inflations, 0.01, 0.80)

            # Projected movements — flat trend
            years_from_base = year - 2025
            projected_movements = base_movements * ((1 + cagr_movements) ** years_from_base)

            # Adjusted rates grow by g annually
            adjusted_charge = avg_charge_per_flight * ((1 + g) ** years_from_base)
            nominal_revenue = projected_movements * adjusted_charge

            # Required revenue = full inflation indexing benchmark
            future_A_t = np.prod(1 + future_inflations)
            full_A_t = A_base * future_A_t
            R_r = projected_movements * (avg_charge_per_flight * full_A_t)

            G = R_r - nominal_revenue
            gap_pct = G / R_r

            simulated_nominal.append(nominal_revenue)
            simulated_required.append(R_r)
            simulated_gaps.append(G)
            gap_pcts.append(gap_pct)

        gaps_array = np.array(simulated_gaps)
        gap_pcts_array = np.array(gap_pcts)

        yearly_results[year] = {
            'mean_nominal': np.mean(simulated_nominal),
            'mean_required': np.mean(simulated_required),
            'mean_gap': np.mean(gaps_array),
            'std_gap': np.std(gaps_array),
            'gap_5th': np.percentile(gaps_array, 5),
            'gap_95th': np.percentile(gaps_array, 95),
            'gap_percent': np.mean(gap_pcts_array) * 100,
            'prob_unsustainable': np.mean(gap_pcts_array > SUSTAINABILITY_THRESHOLD) * 100,
            'prob_sustain': np.mean(gap_pcts_array <= SUSTAINABILITY_THRESHOLD) * 100,
        }

    scenario_results[scenario_name] = yearly_results
    total_gap = sum([yearly_results[yr]['mean_gap'] for yr in proj_years])
    print(f"\n{scenario_name} (g={g*100:.0f}%)")
    print("=" * 90)
    print(f"{'Year':<8} {'Nominal(₦M)':<14} {'Required(₦M)':<16} {'Gap(₦M)':<12} "
          f"{'Gap%':<8} {'P(unsustain)':<15} {'90% CI (₦M)'}")
    print("-" * 90)
    for year, res in yearly_results.items():
        print(f"{year:<8} ₦{res['mean_nominal']/1e6:<12.2f} ₦{res['mean_required']/1e6:<14.2f} "
              f"₦{res['mean_gap']/1e6:<10.2f} {res['gap_percent']:<8.1f} "
              f"{res['prob_unsustainable']:<15.1f}% "
              f"₦{res['gap_5th']/1e6:.1f}M–₦{res['gap_95th']/1e6:.1f}M")
    print(f"Total gap 2026-2030: ₦{total_gap/1e6:.1f} Million")


In [ ]:
# ── CELL 13: Traffic Growth Scenario (S5) with MC Risk Metrics ──
np.random.seed(42)
N_ITERATIONS = 10000
gamma = 0.15  # 15% annual traffic growth

traffic_results = {}

for year in proj_years:
    simulated_nominal = []
    simulated_required = []
    simulated_gaps = []
    gap_pcts = []

    for i in range(N_ITERATIONS):
        n_future_years = year - 2025
        future_inflations = np.random.normal(mean_future_inflation,
                                              std_future_inflation,
                                              n_future_years)
        future_inflations = np.clip(future_inflations, 0.01, 0.80)

        # Traffic growth: movements grow by gamma, rates frozen
        years_from_base = year - 2025
        projected_movements = base_movements * ((1 + gamma) ** years_from_base)
        nominal_revenue = projected_movements * avg_charge_per_flight

        # Required revenue
        future_A_t = np.prod(1 + future_inflations)
        full_A_t = A_base * future_A_t
        R_r = projected_movements * (avg_charge_per_flight * full_A_t)

        G = R_r - nominal_revenue
        gap_pct = G / R_r

        simulated_nominal.append(nominal_revenue)
        simulated_required.append(R_r)
        simulated_gaps.append(G)
        gap_pcts.append(gap_pct)

    gaps_array = np.array(simulated_gaps)
    gap_pcts_array = np.array(gap_pcts)

    traffic_results[year] = {
        'mean_nominal': np.mean(simulated_nominal),
        'mean_required': np.mean(simulated_required),
        'mean_gap': np.mean(gaps_array),
        'std_gap': np.std(gaps_array),
        'gap_5th': np.percentile(gaps_array, 5),
        'gap_95th': np.percentile(gaps_array, 95),
        'gap_percent': np.mean(gap_pcts_array) * 100,
        'prob_unsustainable': np.mean(gap_pcts_array > SUSTAINABILITY_THRESHOLD) * 100,
        'prob_sustain': np.mean(gap_pcts_array <= SUSTAINABILITY_THRESHOLD) * 100,
    }

total_gap_traffic = sum([traffic_results[yr]['mean_gap'] for yr in proj_years])

print("SCENARIO 5 — TRAFFIC GROWTH ONLY (γ=15%, Rates Frozen)")
print("=" * 90)
print(f"{'Year':<8} {'Nominal(₦M)':<14} {'Required(₦M)':<16} {'Gap(₦M)':<12} "
      f"{'Gap%':<8} {'P(unsustain)':<15} {'90% CI (₦M)'}")
print("-" * 90)
for year, res in traffic_results.items():
    print(f"{year:<8} ₦{res['mean_nominal']/1e6:<12.2f} ₦{res['mean_required']/1e6:<14.2f} "
          f"₦{res['mean_gap']/1e6:<10.2f} {res['gap_percent']:<8.1f} "
          f"{res['prob_unsustainable']:<15.1f}% "
          f"₦{res['gap_5th']/1e6:.1f}M–₦{res['gap_95th']/1e6:.1f}M")

print(f"\nTotal gap 2026-2030: ₦{total_gap_traffic/1e6:.1f} Million")
print(f"\nComparison vs Baseline:")
print(f"Baseline total gap:       ₦{total_gap_s1/1e6:.1f}M")
print(f"Traffic growth total gap: ₦{total_gap_traffic/1e6:.1f}M")
print(f"Difference:               ₦{(total_gap_traffic - total_gap_s1)/1e6:.1f}M")
print(f"\nConclusion: Traffic growth alone {'WIDENS' if total_gap_traffic > total_gap_s1 else 'REDUCES'} the absolute gap")


In [ ]:
# ── CELL 14: Combined Scenario Comparison Chart ──

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 11))

# Data preparation
s1_nominal = [baseline_results[yr]['mean_nominal']/1e6 for yr in proj_years]
s1_gap_pct = [baseline_results[yr]['gap_percent'] for yr in proj_years]
s1_gap_5th = [baseline_results[yr]['gap_5th']/1e6 for yr in proj_years]
s1_gap_95th = [baseline_results[yr]['gap_95th']/1e6 for yr in proj_years]

s2_nominal = [scenario_results['S2_Conservative'][yr]['mean_nominal']/1e6 for yr in proj_years]
s2_gap_pct = [scenario_results['S2_Conservative'][yr]['gap_percent'] for yr in proj_years]

s3_nominal = [scenario_results['S3_Moderate'][yr]['mean_nominal']/1e6 for yr in proj_years]
s3_gap_pct = [scenario_results['S3_Moderate'][yr]['gap_percent'] for yr in proj_years]

s4_nominal = [scenario_results['S4_Aggressive'][yr]['mean_nominal']/1e6 for yr in proj_years]
s4_gap_pct = [scenario_results['S4_Aggressive'][yr]['gap_percent'] for yr in proj_years]

s5_nominal = [traffic_results[yr]['mean_nominal']/1e6 for yr in proj_years]
s5_gap_pct = [traffic_results[yr]['gap_percent'] for yr in proj_years]

# Required revenue (same across all price scenarios)
required = [baseline_results[yr]['mean_required']/1e6 for yr in proj_years]

# ── Top chart: Nominal Revenue ──
ax1.plot(proj_years, required, 'k--', linewidth=2, label='Inflation-Required Revenue')
ax1.plot(proj_years, s1_nominal, 'ro-', linewidth=2, markersize=7,
         label='S1: Baseline (Frozen)')
ax1.plot(proj_years, s2_nominal, 'yo-', linewidth=2, markersize=7,
         label='S2: Conservative (g=15%)')
ax1.plot(proj_years, s3_nominal, 'bo-', linewidth=2, markersize=7,
         label='S3: Moderate (g=30%)')
ax1.plot(proj_years, s4_nominal, 'go-', linewidth=2, markersize=7,
         label='S4: Aggressive (g=50%)')
ax1.plot(proj_years, s5_nominal, 'mo-', linewidth=2, markersize=7,
         label='S5: Traffic Growth (γ=15%)')

ax1.fill_between(proj_years, s1_gap_5th,
                 [r + s1_gap_95th[i] - s1_gap_5th[i] for i, r in enumerate(s1_nominal)],
                 alpha=0.1, color='red', label='S1 90% CI')

ax1.set_title('Projected Annual Revenue Under Five Scenarios (2026–2030)', fontsize=13)
ax1.set_xlabel('Year')
ax1.set_ylabel('Revenue (₦ Millions)')
ax1.legend(loc='upper left', fontsize=9)
ax1.set_xticks(proj_years)
ax1.grid(True, alpha=0.3)

# ── Bottom chart: Gap % ──
ax2.plot(proj_years, s1_gap_pct, 'ro-', linewidth=2, markersize=7,
         label='S1: Baseline (Frozen)')
ax2.plot(proj_years, s2_gap_pct, 'yo-', linewidth=2, markersize=7,
         label='S2: Conservative (g=15%)')
ax2.plot(proj_years, s3_gap_pct, 'bo-', linewidth=2, markersize=7,
         label='S3: Moderate (g=30%)')
ax2.plot(proj_years, s4_gap_pct, 'go-', linewidth=2, markersize=7,
         label='S4: Aggressive (g=50%)')
ax2.plot(proj_years, s5_gap_pct, 'mo-', linewidth=2, markersize=7,
         label='S5: Traffic Growth (γ=15%)')

# Sustainability threshold line
ax2.axhline(y=80, color='black', linestyle='--', linewidth=1.5,
            label='Sustainability Threshold (80%)')

# Shade unsustainable zone
ax2.fill_between(proj_years, 80, 100, alpha=0.05, color='red')
ax2.text(2026.1, 81, 'Unsustainable Zone', fontsize=9, color='red', alpha=0.7)

ax2.set_title('Revenue Gap % Under Five Scenarios (2026–2030)', fontsize=13)
ax2.set_xlabel('Year')
ax2.set_ylabel('Gap as % of Inflation-Required Revenue')
ax2.legend(loc='center right', fontsize=9)
ax2.set_xticks(proj_years)
ax2.set_ylim(55, 100)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# ── CELL 15: Fan Charts — All Scenarios ──
np.random.seed(42)
N_ITERATIONS = 10000

# Collect full distribution for all scenarios
fan_data = {
    'S1_Baseline': {'g': 0, 'color': 'red', 'label': 'S1: Baseline (Frozen)'},
    'S2_Conservative': {'g': 0.15, 'color': 'gold', 'label': 'S2: Conservative (g=15%)'},
    'S3_Moderate': {'g': 0.30, 'color': 'blue', 'label': 'S3: Moderate (g=30%)'},
    'S4_Aggressive': {'g': 0.50, 'color': 'green', 'label': 'S4: Aggressive (g=50%)'},
    'S5_Traffic': {'g': 0, 'color': 'purple', 'label': 'S5: Traffic Growth (γ=15%)'},
}

# Store percentiles for each scenario and year
fan_results = {}

for scenario_name, params in fan_data.items():
    g = params['g']
    yearly_percentiles = {}

    for year in proj_years:
        simulated_nominal = []

        for i in range(N_ITERATIONS):
            n_future_years = year - 2025
            future_inflations = np.random.normal(mean_future_inflation,
                                                  std_future_inflation,
                                                  n_future_years)
            future_inflations = np.clip(future_inflations, 0.01, 0.80)

            years_from_base = year - 2025

            if scenario_name == 'S5_Traffic':
                projected_movements = base_movements * ((1 + 0.15) ** years_from_base)
                nominal_revenue = projected_movements * avg_charge_per_flight
            else:
                projected_movements = base_movements * ((1 + cagr_movements) ** years_from_base)
                adjusted_charge = avg_charge_per_flight * ((1 + g) ** years_from_base)
                nominal_revenue = projected_movements * adjusted_charge

            # Required revenue
            future_A_t = np.prod(1 + future_inflations)
            full_A_t = A_base * future_A_t
            R_r = projected_movements * (avg_charge_per_flight * full_A_t)

            # Gap percent
            gap_pct = (R_r - nominal_revenue) / R_r * 100
            simulated_nominal.append(gap_pct)

        arr = np.array(simulated_nominal)
        yearly_percentiles[year] = {
            'mean': np.mean(arr),
            'p5': np.percentile(arr, 5),
            'p10': np.percentile(arr, 10),
            'p25': np.percentile(arr, 25),
            'p50': np.percentile(arr, 50),
            'p75': np.percentile(arr, 75),
            'p90': np.percentile(arr, 90),
            'p95': np.percentile(arr, 95),
        }

    fan_results[scenario_name] = yearly_percentiles
    print(f"{scenario_name} fan data computed ✅")

print("\nAll fan data ready")


In [ ]:
# ── CELL 16 (REVISED): Fan Charts — Cosmetic Fixes ──

# ── Build fan_colors/fan_labels from fan_data (self-contained, no order dependency) ──
fan_colors = {name: params['color'] for name, params in fan_data.items()}
fan_labels = {name: params['label'] for name, params in fan_data.items()}

# ── OPTION B: Combined Fan Chart ──
fig, ax = plt.subplots(figsize=(13, 8))

for scenario_name, yearly_data in fan_results.items():
    color = fan_colors[scenario_name]
    means = [yearly_data[yr]['mean'] for yr in proj_years]
    p5 = [yearly_data[yr]['p5'] for yr in proj_years]
    p95 = [yearly_data[yr]['p95'] for yr in proj_years]

    ax.plot(proj_years, means, color=color, linewidth=2,
            marker='o', markersize=6, label=fan_labels[scenario_name])
    ax.fill_between(proj_years, p5, p95, alpha=0.12, color=color)

# Sustainability threshold
ax.axhline(y=80, color='black', linestyle='--', linewidth=1.5,
           label='Sustainability Threshold (80%)')

# Shade unsustainable zone — top portion
ax.fill_between(proj_years, 80, 105, alpha=0.04, color='red')
ax.text(2029.2, 97, 'Unsustainable\nZone', fontsize=9, 
        color='red', alpha=0.8, ha='center')

# Shade sustainable zone — bottom portion
ax.fill_between(proj_years, 50, 80, alpha=0.04, color='green')
ax.text(2029.2, 55, 'Sustainable\nZone', fontsize=9,
        color='green', alpha=0.8, ha='center')

ax.set_title('Fan Chart: Revenue Gap % Under Five Scenarios (2026–2030)\n'
             'Shaded bands = 90% Confidence Interval', fontsize=13)
ax.set_xlabel('Year')
ax.set_ylabel('Gap as % of Inflation-Required Revenue')
ax.legend(loc='upper left', fontsize=9)
ax.set_xticks(proj_years)
ax.set_ylim(50, 105)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── OPTION A: Individual Fan Charts ──
fig, axes = plt.subplots(2, 3, figsize=(16, 11))
axes = axes.flatten()

for idx, (scenario_name, yearly_data) in enumerate(fan_results.items()):
    ax = axes[idx]
    color = fan_colors[scenario_name]

    means = [yearly_data[yr]['mean'] for yr in proj_years]
    p5  = [yearly_data[yr]['p5']  for yr in proj_years]
    p10 = [yearly_data[yr]['p10'] for yr in proj_years]
    p25 = [yearly_data[yr]['p25'] for yr in proj_years]
    p75 = [yearly_data[yr]['p75'] for yr in proj_years]
    p90 = [yearly_data[yr]['p90'] for yr in proj_years]
    p95 = [yearly_data[yr]['p95'] for yr in proj_years]

    # Three bands
    ax.fill_between(proj_years, p5,  p95, alpha=0.15, color=color, label='90% CI')
    ax.fill_between(proj_years, p10, p90, alpha=0.20, color=color, label='80% CI')
    ax.fill_between(proj_years, p25, p75, alpha=0.35, color=color, label='50% CI')

    # Mean line
    ax.plot(proj_years, means, color=color, linewidth=2.5,
            marker='o', markersize=7, label='Mean', zorder=5)

    # Sustainability threshold
    ax.axhline(y=80, color='black', linestyle='--', linewidth=1.2, alpha=0.8)

    # Shade zones
    ax.fill_between(proj_years, 80, 105, alpha=0.04, color='red')
    ax.fill_between(proj_years, 40, 80, alpha=0.04, color='green')
    ax.text(2026.1, 81.5, '80% threshold', fontsize=7.5,
            color='black', alpha=0.8)

    # Add mean value labels at 2030
    ax.annotate(f"{means[-1]:.1f}%",
                xy=(2030, means[-1]),
                xytext=(5, 0), textcoords='offset points',
                fontsize=8, color=color, fontweight='bold')

    ax.set_title(fan_labels[scenario_name], fontsize=10, fontweight='bold', color=color)
    ax.set_xlabel('Year', fontsize=9)
    ax.set_ylabel('Gap %', fontsize=9)
    ax.set_xticks(proj_years)
    ax.set_ylim(40, 105)
    ax.legend(fontsize=7.5, loc='lower left')
    ax.grid(True, alpha=0.3)

# Hide 6th subplot
axes[5].set_visible(False)

fig.suptitle('Individual Fan Charts: Revenue Gap % by Scenario (2026–2030)\n'
             '50%, 80%, and 90% Confidence Intervals',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── CELL 17: Sustainability Indicators Table ──

print("SUSTAINABILITY INDICATORS — All Scenarios (2026–2030)")
print("=" * 100)

# Standardise all results into one consistent structure
def get_prob_unsustainable(results, year):
    res = results[year]
    if 'prob_unsustainable' in res:
        return res['prob_unsustainable']
    elif 'prob_below_threshold' in res:
        return res['prob_below_threshold']
    else:
        return None

all_scenarios = {
    'S1: Baseline (g=0%)': baseline_results,
    'S2: Conservative (g=15%)': scenario_results['S2_Conservative'],
    'S3: Moderate (g=30%)': scenario_results['S3_Moderate'],
    'S4: Aggressive (g=50%)': scenario_results['S4_Aggressive'],
    'S5: Traffic Growth (γ=15%)': traffic_results
}

current_charge = avg_charge_per_flight

# ── Table 1: Total Gap and Gap Reduction ──
print("\nTABLE 1: Total Revenue Gap and Gap Reduction vs Baseline")
print("-" * 80)
print(f"{'Scenario':<30} {'Total Gap (₦M)':<18} {'Reduction vs S1':<20} {'Reduction %'}")
print("-" * 80)

total_gaps = {}
for name, results in all_scenarios.items():
    total_gap = sum([results[yr]['mean_gap'] for yr in proj_years])
    total_gaps[name] = total_gap

baseline_gap = total_gaps['S1: Baseline (g=0%)']

for name, total_gap in total_gaps.items():
    reduction = baseline_gap - total_gap
    reduction_pct = (reduction / baseline_gap) * 100
    direction = "-" if reduction >= 0 else "+"
    print(f"{name:<30} ₦{total_gap/1e6:<16.1f} "
          f"{direction}₦{abs(reduction)/1e6:<18.1f} {reduction_pct:.1f}%")

# ── Table 2: Probability of Unsustainability by Year ──
print("\nTABLE 2: Probability of Unsustainable Revenue (Gap > 80%) by Year")
print("-" * 80)
print(f"{'Scenario':<30}", end="")
for yr in proj_years:
    print(f"{yr:<12}", end="")
print()
print("-" * 80)

for name, results in all_scenarios.items():
    print(f"{name:<30}", end="")
    for yr in proj_years:
        prob = get_prob_unsustainable(results, yr)
        print(f"{prob:.1f}%{'':<6}", end="")
    print()

# ── Table 3: Implied Charge Per Flight by 2030 ──
print("\nTABLE 3: Implied Charge Per Flight by 2030")
print("-" * 80)
print(f"{'Scenario':<30} {'2026 Charge':<16} {'2028 Charge':<16} "
      f"{'2030 Charge':<16} {'vs Current'}")
print("-" * 80)

g_map = {
    'S1: Baseline (g=0%)': 0,
    'S2: Conservative (g=15%)': 0.15,
    'S3: Moderate (g=30%)': 0.30,
    'S4: Aggressive (g=50%)': 0.50,
    'S5: Traffic Growth (γ=15%)': 0
}

for name, g in g_map.items():
    c2026 = current_charge * ((1 + g) ** 1)
    c2028 = current_charge * ((1 + g) ** 3)
    c2030 = current_charge * ((1 + g) ** 5)
    increase = ((c2030 / current_charge) - 1) * 100
    print(f"{name:<30} ₦{c2026:<14,.0f} ₦{c2028:<14,.0f} "
          f"₦{c2030:<14,.0f} +{increase:.1f}%")

# ── Table 4: First year gap crosses below 80% ──
print("\nTABLE 4: First Year Gap Falls Below 80% Sustainability Threshold")
print("-" * 60)
print(f"{'Scenario':<30} {'First Sustainable Year'}")
print("-" * 60)

for name, results in all_scenarios.items():
    sustainable_year = None
    for yr in proj_years:
        if results[yr]['gap_percent'] < 80:
            sustainable_year = yr
            break
    if sustainable_year:
        print(f"{name:<30} {sustainable_year}")
    else:
        print(f"{name:<30} Never (within 2026-2030)")

print(f"\nCurrent charge per flight: ₦{current_charge:,.0f}")
print(f"Inflation-required charge by 2030 (full indexing): "
      f"₦{current_charge * cumulative_factor_lagged(2031):,.0f}")


In [ ]:
# Deterministic inflation-required charge by 2030
# using historical inflation through 2025 and
# the historical mean inflation rate for 2026-2030.
future_factor = (1 + mean_future_inflation) ** 5
full_factor_2030 = cumulative_factor_lagged(2026) * future_factor
print(f"Inflation-required charge by 2030 (full indexing): "
      f"₦{current_charge * full_factor_2030:,.0f}")


In [ ]:
# ── BATCH EXPORT ALL CHARTS (300 DPI) ──
import os

export_folder = os.path.join(os.path.expanduser("~"), "Documents", "NAMA_Charts")
os.makedirs(export_folder, exist_ok=True)

def save(name):
    plt.savefig(os.path.join(export_folder, name), dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ {name}")

print("Exporting charts...")


# Chart 1: Monthly Revenue
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_monthly['REVENUE']/1e6, color='blue', linewidth=1.5)
ax.set_title('NAMA Yola Monthly Nominal Revenue (Jan 2021 – Sep 2025)', fontsize=13)
ax.set_xlabel('Month Index'); ax.set_ylabel('Revenue (₦ Millions)')
ax.grid(True, alpha=0.3); plt.tight_layout()
save('chart_01_monthly_revenue.png')


# Chart 2: Real Value Erosion
years_list = list(range(2008, 2026))
cum = 1.0
rv = [1.0]

for yr in range(2009, 2026):
    cum *= (1 + inflation_data[yr])
    rv.append(1/cum)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(years_list, rv, color='red', marker='o', linewidth=2)
ax.fill_between(years_list, rv, 1, alpha=0.1, color='red')
ax.set_title('Real Value of ₦1 of 2008 NAMA Yola Revenue (2008–2025)', fontsize=13)
ax.set_xlabel('Year')
ax.set_ylabel('Real Value (₦, 2008 base)')
ax.axhline(y=rv[-1], color='gray', linestyle='--', linewidth=1)
ax.annotate(
    '₦0.09 remaining\n(90.9% lost)',
    xy=(2025, rv[-1]),
    xytext=(2019, 0.25),
    arrowprops=dict(arrowstyle='->', color='black'),
    fontsize=10,
    color='red'
)
plt.xticks(years_list, rotation=45)
plt.tight_layout()
save('chart_02_real_value_erosion.png')


# Chart 3: Retrospective Gap
yrs  = annual_actual['YEAR'].astype(int).values
act  = annual_actual['ACTUAL_REVENUE'].values
adj  = annual_actual['ADJUSTED_REVENUE'].values
gap  = annual_actual['REVENUE_GAP'].values
gpct = annual_actual['GAP_PERCENT'].values

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
ax1.bar(yrs-0.2, act/1e6, width=0.4, label='Actual Revenue (Nominal)', color='steelblue')
ax1.bar(yrs+0.2, adj/1e6, width=0.4, label='Inflation-Adjusted Revenue', color='red', alpha=0.7)
ax1.set_title('NAMA Yola: Actual vs Inflation-Adjusted Revenue (2021–2025)', fontsize=13)
ax1.set_xlabel('Year')
ax1.set_ylabel('Revenue (₦ Millions)')
ax1.legend()
ax1.set_xticks(yrs)

ax2.bar(yrs, gap/1e6, color='darkred', alpha=0.8)
ax2.set_title('Annual Revenue Gap — Purchasing Power Lost (2021–2025)', fontsize=13)
ax2.set_xlabel('Year')
ax2.set_ylabel('Gap (₦ Millions)')
ax2.set_xticks(yrs)

for y, g, p in zip(yrs, gap, gpct):
    ax2.text(y, g/1e6+1, f'{p:.1f}%', ha='center', fontsize=10, color='darkred')

plt.tight_layout()
save('chart_03_retrospective_gap.png')


# Chart 4: MC Retrospective
np.random.seed(42)
mc_results_retro = {}

for year in [2021, 2022, 2023, 2024, 2025]:
    actual_rev = annual_actual.loc[
        annual_actual['YEAR'] == year,
        'ACTUAL_REVENUE'
    ].values[0]

    gaps = []
    reals = []

    for _ in range(N_ITERATIONS):
        At = 1.0

        for yr in range(2008, year):
            r = np.random.normal(inflation_data[yr], 0.05)
            At *= (1 + np.clip(r, 0.01, 0.80))

        Rr = actual_rev * At
        gaps.append(Rr - actual_rev)
        reals.append(Rr)

    mc_results_retro[year] = {
        'actual_revenue': actual_rev,
        'mean_real_revenue': np.mean(reals),
        'mean_gap': np.mean(gaps),
        'gap_5th': np.percentile(gaps, 5),
        'gap_95th': np.percentile(gaps, 95),
        'gap_percent': (np.mean(gaps) / np.mean(reals)) * 100
    }

det_gaps = {
    yr: annual_actual.loc[
        annual_actual['YEAR'] == yr,
        'REVENUE_GAP'
    ].values[0]
    for yr in [2021, 2022, 2023, 2024, 2025]
}

yrl = list(mc_results_retro.keys())
actr = [mc_results_retro[y]['actual_revenue']/1e6 for y in yrl]
mr   = [mc_results_retro[y]['mean_real_revenue']/1e6 for y in yrl]
mg   = [mc_results_retro[y]['mean_gap']/1e6 for y in yrl]
g5   = [mc_results_retro[y]['gap_5th']/1e6 for y in yrl]
g95  = [mc_results_retro[y]['gap_95th']/1e6 for y in yrl]
dv   = [det_gaps[y]/1e6 for y in yrl]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

ax1.plot(yrl, actr, 'bo-', linewidth=2, markersize=8,
         label='Actual Nominal Revenue')
ax1.plot(yrl, mr, 'ro-', linewidth=2, markersize=8,
         label='MC Mean Inflation-Required Revenue')

ax1.fill_between(
    yrl,
    [g5[i] + actr[i] for i in range(len(yrl))],
    [g95[i] + actr[i] for i in range(len(yrl))],
    alpha=0.2,
    color='red',
    label='90% CI'
)

ax1.set_title(
    'MC Retrospective: Actual vs Inflation-Required Revenue (2021–2025)',
    fontsize=12
)
ax1.set_xlabel('Year')
ax1.set_ylabel('Revenue (₦ Millions)')
ax1.legend()
ax1.set_xticks(yrl)

ax2.plot(yrl, mg, 'ro-', linewidth=2, markersize=8,
         label='MC Mean Gap')
ax2.fill_between(yrl, g5, g95, alpha=0.2, color='red', label='90% CI')
ax2.plot(yrl, g5, 'r--', linewidth=1, alpha=0.5)
ax2.plot(yrl, g95, 'r--', linewidth=1, alpha=0.5)

ax2.scatter(
    yrl, dv,
    color='darkred',
    zorder=5,
    s=80,
    marker='D',
    label='Deterministic Gap'
)

for y, g in zip(yrl, mg):
    ax2.annotate(
        f'₦{g:.1f}M',
        xy=(y, g),
        xytext=(0, 10),
        textcoords='offset points',
        ha='center',
        fontsize=9,
        color='darkred'
    )

ax2.set_title(
    'MC Revenue Gap with 90% Confidence Bands (2021–2025)',
    fontsize=12
)
ax2.set_xlabel('Year')
ax2.set_ylabel('Revenue Gap (₦ Millions)')
ax2.legend()
ax2.set_xticks(yrl)

plt.tight_layout()
save('chart_04_mc_retrospective.png')


# Chart 5: Sensitivity Analysis
np.random.seed(42)
sensitivity_results = {}

for g in g_values:
    yr_res = {}

    for year in proj_years:
        nom = []
        req = []
        gaps = []

        for _ in range(N_ITERATIONS):
            nf = year - 2025
            fi = np.clip(
                np.random.normal(
                    mean_future_inflation,
                    std_future_inflation,
                    nf
                ),
                0.01,
                0.80
            )

            yfb = year - 2025
            pm = base_movements * ((1 + cagr_movements) ** yfb)
            ac = avg_charge_per_flight * ((1 + g) ** yfb)
            nr = pm * ac

            fAt = np.prod(1 + fi)
            fA = A_base * fAt
            Rr = pm * (avg_charge_per_flight * fA)

            nom.append(nr)
            req.append(Rr)
            gaps.append(Rr - nr)

        yr_res[year] = {
            'mean_nominal': np.mean(nom),
            'mean_required': np.mean(req),
            'mean_gap': np.mean(gaps),
            'gap_percent': (np.mean(gaps) / np.mean(req)) * 100
        }

    sensitivity_results[g] = yr_res

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

colors_s = plt.cm.RdYlGn_r(
    np.linspace(0.1, 0.9, len(g_values))
)

for i, g in enumerate(g_values):
    ax1.plot(
        proj_years,
        [sensitivity_results[g][y]['gap_percent'] for y in proj_years],
        marker='o',
        color=colors_s[i],
        linewidth=2,
        label=f'g={g*100:.0f}%'
    )

ax1.set_title(
    'Revenue Gap % by Year\nUnder Different Adjustment Rates',
    fontsize=12
)
ax1.set_xlabel('Year')
ax1.set_ylabel('Gap %')
ax1.legend(loc='upper left', fontsize=9)
ax1.set_xticks(proj_years)
ax1.grid(True, alpha=0.3)

for year in proj_years:
    ax2.plot(
        [g*100 for g in g_values],
        [sensitivity_results[g][year]['gap_percent'] for g in g_values],
        marker='o',
        linewidth=2,
        label=str(year)
    )

ax2.set_title(
    'Revenue Gap % by Adjustment Rate\nFor Each Year',
    fontsize=12
)
ax2.set_xlabel('Annual Adjustment Rate (%)')
ax2.set_ylabel('Gap %')
ax2.legend(title='Year', fontsize=9)
ax2.set_xticks([g*100 for g in g_values])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
save('chart_05_sensitivity_analysis.png')


# Chart 6: Scenario Comparison
np.random.seed(42)

def run_scenario(g_val, gamma=None):
    res = {}

    for year in proj_years:
        nom = []
        req = []
        gaps = []
        gpcts = []

        for _ in range(N_ITERATIONS):
            nf = year - 2025

            fi = np.clip(
                np.random.normal(
                    mean_future_inflation,
                    std_future_inflation,
                    nf
                ),
                0.01,
                0.80
            )

            yfb = year - 2025

            if gamma:
                pm = base_movements * ((1 + gamma) ** yfb)
                nr = pm * avg_charge_per_flight
            else:
                pm = base_movements * ((1 + cagr_movements) ** yfb)
                nr = pm * (
                    avg_charge_per_flight *
                    ((1 + g_val) ** yfb)
                )

            fAt = np.prod(1 + fi)
            fA = A_base * fAt

            Rr = pm * (avg_charge_per_flight * fA)

            G = Rr - nr
            gp = G / Rr

            nom.append(nr)
            req.append(Rr)
            gaps.append(G)
            gpcts.append(gp)

        ga = np.array(gaps)
        gpa = np.array(gpcts)

        res[year] = {
            'mean_nominal': np.mean(nom),
            'mean_required': np.mean(req),
            'mean_gap': np.mean(ga),
            'gap_5th': np.percentile(ga, 5),
            'gap_95th': np.percentile(ga, 95),
            'gap_percent': np.mean(gpa) * 100,
            'prob_unsustainable': np.mean(
                gpa > SUSTAINABILITY_THRESHOLD
            ) * 100,
            'prob_sustain': np.mean(
                gpa <= SUSTAINABILITY_THRESHOLD
            ) * 100
        }

    return res


baseline_results = run_scenario(0)

scenario_results = {
    'S2_Conservative': run_scenario(0.15),
    'S3_Moderate': run_scenario(0.30),
    'S4_Aggressive': run_scenario(0.50)
}

traffic_results = run_scenario(0, gamma=0.15)

s1n = [baseline_results[y]['mean_nominal']/1e6 for y in proj_years]
s2n = [scenario_results['S2_Conservative'][y]['mean_nominal']/1e6 for y in proj_years]
s3n = [scenario_results['S3_Moderate'][y]['mean_nominal']/1e6 for y in proj_years]
s4n = [scenario_results['S4_Aggressive'][y]['mean_nominal']/1e6 for y in proj_years]
s5n = [traffic_results[y]['mean_nominal']/1e6 for y in proj_years]

req = [baseline_results[y]['mean_required']/1e6 for y in proj_years]

s1g = [baseline_results[y]['gap_percent'] for y in proj_years]
s2g = [scenario_results['S2_Conservative'][y]['gap_percent'] for y in proj_years]
s3g = [scenario_results['S3_Moderate'][y]['gap_percent'] for y in proj_years]
s4g = [scenario_results['S4_Aggressive'][y]['gap_percent'] for y in proj_years]
s5g = [traffic_results[y]['gap_percent'] for y in proj_years]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 11))

ax1.plot(
    proj_years, req, 'k--',
    linewidth=2,
    label='Inflation-Required Revenue'
)
ax1.plot(
    proj_years, s1n, 'ro-',
    linewidth=2, markersize=7,
    label='S1: Baseline'
)
ax1.plot(
    proj_years, s2n, 'yo-',
    linewidth=2, markersize=7,
    label='S2: Conservative (g=15%)'
)
ax1.plot(
    proj_years, s3n, 'bo-',
    linewidth=2, markersize=7,
    label='S3: Moderate (g=30%)'
)
ax1.plot(
    proj_years, s4n, 'go-',
    linewidth=2, markersize=7,
    label='S4: Aggressive (g=50%)'
)
ax1.plot(
    proj_years, s5n, 'mo-',
    linewidth=2, markersize=7,
    label='S5: Traffic Growth (γ=15%)'
)

ax1.set_title(
    'Projected Annual Revenue Under Five Scenarios (2026–2030)',
    fontsize=13
)
ax1.set_xlabel('Year')
ax1.set_ylabel('Revenue (₦ Millions)')
ax1.legend(loc='upper left', fontsize=9)
ax1.set_xticks(proj_years)
ax1.grid(True, alpha=0.3)

ax2.plot(
    proj_years, s1g, 'ro-',
    linewidth=2, markersize=7,
    label='S1: Baseline'
)
ax2.plot(
    proj_years, s2g, 'yo-',
    linewidth=2, markersize=7,
    label='S2: Conservative (g=15%)'
)
ax2.plot(
    proj_years, s3g, 'bo-',
    linewidth=2, markersize=7,
    label='S3: Moderate (g=30%)'
)
ax2.plot(
    proj_years, s4g, 'go-',
    linewidth=2, markersize=7,
    label='S4: Aggressive (g=50%)'
)
ax2.plot(
    proj_years, s5g, 'mo-',
    linewidth=2, markersize=7,
    label='S5: Traffic Growth (γ=15%)'
)

ax2.axhline(
    y=80,
    color='black',
    linestyle='--',
    linewidth=1.5,
    label='80% Threshold'
)

ax2.fill_between(
    proj_years, 80, 105,
    alpha=0.04,
    color='red'
)

ax2.fill_between(
    proj_years, 50, 80,
    alpha=0.04,
    color='green'
)

ax2.text(
    2029.2, 97,
    'Unsustainable Zone',
    fontsize=9,
    color='red',
    alpha=0.8,
    ha='center'
)

ax2.text(
    2029.2, 55,
    'Sustainable Zone',
    fontsize=9,
    color='green',
    alpha=0.8,
    ha='center'
)

ax2.set_title(
    'Revenue Gap % Under Five Scenarios (2026–2030)',
    fontsize=13
)
ax2.set_xlabel('Year')
ax2.set_ylabel('Gap %')
ax2.legend(loc='upper left', fontsize=9)
ax2.set_xticks(proj_years)
ax2.set_ylim(50, 105)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
save('chart_06_scenario_comparison.png')


# Charts 7 & 8: Fan Charts
np.random.seed(42)

colors = {
    'S1_Baseline': 'red',
    'S2_Conservative': 'goldenrod',
    'S3_Moderate': 'blue',
    'S4_Aggressive': 'green',
    'S5_Traffic': 'purple'
}

labels = {
    'S1_Baseline': 'S1: Baseline (Frozen)',
    'S2_Conservative': 'S2: Conservative (g=15%)',
    'S3_Moderate': 'S3: Moderate (g=30%)',
    'S4_Aggressive': 'S4: Aggressive (g=50%)',
    'S5_Traffic': 'S5: Traffic Growth (γ=15%)'
}

sc_map = {
    'S1_Baseline': (0, None),
    'S2_Conservative': (0.15, None),
    'S3_Moderate': (0.30, None),
    'S4_Aggressive': (0.50, None),
    'S5_Traffic': (0, 0.15)
}

fan_results = {}

for sn, (g, gm) in sc_map.items():
    ydata = {}

    for year in proj_years:
        arr = []

        for _ in range(N_ITERATIONS):
            nf = year - 2025

            fi = np.clip(
                np.random.normal(
                    mean_future_inflation,
                    std_future_inflation,
                    nf
                ),
                0.01,
                0.80
            )

            yfb = year - 2025

            if gm:
                pm = base_movements * ((1 + gm) ** yfb)
                nr = pm * avg_charge_per_flight
            else:
                pm = base_movements * ((1 + cagr_movements) ** yfb)
                nr = pm * (
                    avg_charge_per_flight *
                    ((1 + g) ** yfb)
                )

            fAt = np.prod(1 + fi)
            fA = A_base * fAt

            Rr = pm * (avg_charge_per_flight * fA)

            arr.append((Rr - nr) / Rr * 100)

        a = np.array(arr)

        ydata[year] = {
            'mean': np.mean(a),
            'p5': np.percentile(a, 5),
            'p10': np.percentile(a, 10),
            'p25': np.percentile(a, 25),
            'p50': np.percentile(a, 50),
            'p75': np.percentile(a, 75),
            'p90': np.percentile(a, 90),
            'p95': np.percentile(a, 95)
        }

    fan_results[sn] = ydata
    print(f"Fan {sn} ✅")


# Chart 7: Combined
fig, ax = plt.subplots(figsize=(13, 8))

for sn, yd in fan_results.items():
    c = colors[sn]

    ax.plot(
        proj_years,
        [yd[y]['mean'] for y in proj_years],
        color=c,
        linewidth=2,
        marker='o',
        markersize=6,
        label=labels[sn]
    )

    ax.fill_between(
        proj_years,
        [yd[y]['p5'] for y in proj_years],
        [yd[y]['p95'] for y in proj_years],
        alpha=0.12,
        color=c
    )

ax.axhline(
    y=80,
    color='black',
    linestyle='--',
    linewidth=1.5,
    label='80% Threshold'
)

ax.fill_between(
    proj_years, 80, 105,
    alpha=0.04,
    color='red'
)

ax.fill_between(
    proj_years, 50, 80,
    alpha=0.04,
    color='green'
)

ax.text(
    2029.2, 97,
    'Unsustainable\nZone',
    fontsize=9,
    color='red',
    alpha=0.8,
    ha='center'
)

ax.text(
    2029.2, 55,
    'Sustainable\nZone',
    fontsize=9,
    color='green',
    alpha=0.8,
    ha='center'
)

ax.set_title(
    'Fan Chart: Revenue Gap % Under Five Scenarios (2026–2030)\n'
    'Shaded = 90% CI',
    fontsize=13
)

ax.set_xlabel('Year')
ax.set_ylabel('Gap %')
ax.legend(loc='upper left', fontsize=9)
ax.set_xticks(proj_years)
ax.set_ylim(50, 105)
ax.grid(True, alpha=0.3)

plt.tight_layout()
save('chart_07_combined_fan_chart.png')


# Chart 8: Individual
fig, axes = plt.subplots(2, 3, figsize=(16, 11))
axes = axes.flatten()

for idx, (sn, yd) in enumerate(fan_results.items()):
    ax = axes[idx]
    c = colors[sn]

    ax.fill_between(
        proj_years,
        [yd[y]['p5'] for y in proj_years],
        [yd[y]['p95'] for y in proj_years],
        alpha=0.15,
        color=c,
        label='90% CI'
    )

    ax.fill_between(
        proj_years,
        [yd[y]['p10'] for y in proj_years],
        [yd[y]['p90'] for y in proj_years],
        alpha=0.20,
        color=c,
        label='80% CI'
    )

    ax.fill_between(
        proj_years,
        [yd[y]['p25'] for y in proj_years],
        [yd[y]['p75'] for y in proj_years],
        alpha=0.35,
        color=c,
        label='50% CI'
    )

    means = [yd[y]['mean'] for y in proj_years]

    ax.plot(
        proj_years,
        means,
        color=c,
        linewidth=2.5,
        marker='o',
        markersize=7,
        label='Mean',
        zorder=5
    )

    ax.axhline(
        y=80,
        color='black',
        linestyle='--',
        linewidth=1.2,
        alpha=0.8
    )

    ax.fill_between(
        proj_years, 80, 105,
        alpha=0.04,
        color='red'
    )

    ax.fill_between(
        proj_years, 40, 80,
        alpha=0.04,
        color='green'
    )

    ax.text(
        2026.1, 81.5,
        '80% threshold',
        fontsize=7.5,
        color='black',
        alpha=0.8
    )

    ax.annotate(
        f"{means[-1]:.1f}%",
        xy=(2030, means[-1]),
        xytext=(5, 0),
        textcoords='offset points',
        fontsize=8,
        color=c,
        fontweight='bold'
    )

    ax.set_title(
        labels[sn],
        fontsize=10,
        fontweight='bold',
        color=c
    )

    ax.set_xlabel('Year', fontsize=9)
    ax.set_ylabel('Gap %', fontsize=9)
    ax.set_xticks(proj_years)
    ax.set_ylim(40, 105)
    ax.legend(fontsize=7.5, loc='lower left')
    ax.grid(True, alpha=0.3)

axes[5].set_visible(False)

fig.suptitle(
    'Individual Fan Charts: Revenue Gap % by Scenario (2026–2030)\n'
    '50%, 80%, and 90% CIs',
    fontsize=13,
    fontweight='bold',
    y=1.01
)

plt.tight_layout()
save('chart_08_individual_fan_charts.png')


# ── EXPORT COMPLETE ──
print(f"\n✅ All 8 charts exported to:")
print(export_folder)


In [ ]:
# ── MASTER SETUP CELL — Run this first after any session reset ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import os

# ── Load data ──
df_flights = pd.read_excel("DATA.xlsx", sheet_name="Flight Records")
df_monthly = pd.read_excel("DATA.xlsx", sheet_name="Monthly Summary")
df_annual  = pd.read_excel("DATA.xlsx", sheet_name="Annual Summary")

for col in ['TOTAL','AUW','YEAR','MONTH']:
    if col in df_flights.columns:
        df_flights[col] = pd.to_numeric(df_flights[col], errors='coerce')
df_flights = df_flights[df_flights['TOTAL'] > 0].reset_index(drop=True)

df_monthly['REVENUE'] = pd.to_numeric(df_monthly['REVENUE'], errors='coerce')
df_monthly['YEAR']    = pd.to_numeric(df_monthly['YEAR'],    errors='coerce')
df_monthly['MONTH']   = pd.to_numeric(df_monthly['MONTH'],   errors='coerce')
df_monthly['FLIGHTS'] = pd.to_numeric(df_monthly['FLIGHTS'], errors='coerce')

annual_revenue = df_flights.groupby('YEAR')['TOTAL'].sum()

# ── Inflation data ──
inflation_data = {
    2008:0.1158, 2009:0.1254, 2010:0.1374, 2011:0.1083,
    2012:0.1222, 2013:0.0850, 2014:0.0805, 2015:0.0901,
    2016:0.1570, 2017:0.1650, 2018:0.1210, 2019:0.1140,
    2020:0.1325, 2021:0.1695, 2022:0.1885, 2023:0.2466,
    2024:0.3324, 2025:0.2333
}
inflation_df = pd.DataFrame(list(inflation_data.items()),
                             columns=['YEAR','INFLATION_RATE'])

def cumulative_factor_lagged(year):
    factor = 1.0
    for yr in range(2008, year):
        if yr in inflation_data:
            factor *= (1 + inflation_data[yr])
    return factor

mean_inflation        = inflation_df['INFLATION_RATE'].mean()
std_inflation         = inflation_df['INFLATION_RATE'].std()
mean_future_inflation = mean_inflation
std_future_inflation  = std_inflation

# ── Simulation base parameters ──
base_revenue      = annual_revenue[2025] * (12/9)
base_movements    = df_monthly[df_monthly['YEAR']==2025]['FLIGHTS'].sum() * (12/9)
avg_charge_per_flight = base_revenue / base_movements
A_base            = cumulative_factor_lagged(2025)

actual_2021 = df_monthly[df_monthly['YEAR']==2021]['ACTUAL_FLIGHTS'].sum()
actual_2024 = df_monthly[df_monthly['YEAR']==2024]['ACTUAL_FLIGHTS'].sum()
cagr_movements = (actual_2024 / actual_2021) ** (1/3) - 1

proj_years = [2026, 2027, 2028, 2029, 2030]
g_values   = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
SUSTAINABILITY_THRESHOLD = 0.80
N_ITERATIONS = 10000

# ── Annual actual revenue for retrospective ──
annual_actual = df_monthly.groupby('YEAR')['REVENUE'].sum().reset_index()
annual_actual.columns = ['YEAR','ACTUAL_REVENUE']
annual_actual.loc[annual_actual['YEAR']==2025,'ACTUAL_REVENUE'] *= (12/9)
anchor_2021 = annual_actual.loc[annual_actual['YEAR']==2021,'ACTUAL_REVENUE'].values[0]
annual_actual['INFLATION_FACTOR'] = annual_actual['YEAR'].apply(
    lambda yr: cumulative_factor_lagged(int(yr)))
annual_actual['ADJUSTED_REVENUE'] = anchor_2021 * annual_actual['INFLATION_FACTOR']
annual_actual['REVENUE_GAP']      = annual_actual['ADJUSTED_REVENUE'] - annual_actual['ACTUAL_REVENUE']
annual_actual['GAP_PERCENT']      = (annual_actual['REVENUE_GAP'] /
                                      annual_actual['ADJUSTED_REVENUE']) * 100

# ── Inflation-required charge by 2030 ──
future_factor      = (1 + mean_future_inflation) ** 5
full_factor_2030   = cumulative_factor_lagged(2026) * future_factor
required_charge_2030 = avg_charge_per_flight * full_factor_2030

print("✅ Setup complete")
print(f"Base revenue:          ₦{base_revenue/1e6:.2f}M")
print(f"Avg charge/flight:     ₦{avg_charge_per_flight:,.0f}")
print(f"Movement CAGR:         {cagr_movements*100:.2f}%")
print(f"A_base (2025):         {A_base:.4f}")
print(f"Mean inflation:        {mean_inflation*100:.2f}%")
print(f"Std inflation:         {std_inflation*100:.2f}%")
print(f"Required charge 2030:  ₦{required_charge_2030:,.0f}")


In [ ]:
# ── SUSTAINABILITY INDICATORS TABLE ──

def get_prob_unsustainable(results, year):
    res = results[year]
    if 'prob_unsustainable' in res:
        return res['prob_unsustainable']
    elif 'prob_below_threshold' in res:
        return res['prob_below_threshold']
    return 0.0

all_scenarios = {
    'S1: Baseline (g=0%)':          baseline_results,
    'S2: Conservative (g=15%)':     scenario_results['S2_Conservative'],
    'S3: Moderate (g=30%)':         scenario_results['S3_Moderate'],
    'S4: Aggressive (g=50%)':       scenario_results['S4_Aggressive'],
    'S5: Traffic Growth (γ=15%)':   traffic_results
}

current_charge = avg_charge_per_flight

# Table 1
print("TABLE 1: Total Revenue Gap and Reduction vs Baseline")
print("-" * 75)
print(f"{'Scenario':<32} {'Total Gap (₦M)':<18} {'Reduction vs S1':<20} {'Reduction %'}")
print("-" * 75)
total_gaps = {name: sum([r[yr]['mean_gap'] for yr in proj_years])
              for name, r in all_scenarios.items()}
baseline_gap = total_gaps['S1: Baseline (g=0%)']
for name, tg in total_gaps.items():
    reduction = baseline_gap - tg
    pct = (reduction / baseline_gap) * 100
    sign = "-" if reduction >= 0 else "+"
    print(f"{name:<32} ₦{tg/1e6:<16.1f} {sign}₦{abs(reduction)/1e6:<18.1f} {pct:.1f}%")

# Table 2
print("\nTABLE 2: Probability of Unsustainable Revenue (Gap > 80%) by Year")
print("-" * 80)
print(f"{'Scenario':<32}", end="")
for yr in proj_years:
    print(f"{yr:<12}", end="")
print()
print("-" * 80)
for name, results in all_scenarios.items():
    print(f"{name:<32}", end="")
    for yr in proj_years:
        print(f"{get_prob_unsustainable(results, yr):.1f}%{'':<6}", end="")
    print()

# Table 3
print("\nTABLE 3: Implied Charge Per Flight")
print("-" * 75)
print(f"{'Scenario':<32} {'2026':<16} {'2028':<16} {'2030':<16} {'vs Current'}")
print("-" * 75)
g_map = {'S1: Baseline (g=0%)': 0, 'S2: Conservative (g=15%)': 0.15,
         'S3: Moderate (g=30%)': 0.30, 'S4: Aggressive (g=50%)': 0.50,
         'S5: Traffic Growth (γ=15%)': 0}
for name, g in g_map.items():
    c2026 = current_charge * ((1+g)**1)
    c2028 = current_charge * ((1+g)**3)
    c2030 = current_charge * ((1+g)**5)
    pct   = ((c2030/current_charge)-1)*100
    print(f"{name:<32} ₦{c2026:<14,.0f} ₦{c2028:<14,.0f} ₦{c2030:<14,.0f} +{pct:.1f}%")

# Table 4
print("\nTABLE 4: First Year Gap Falls Below 80% Threshold")
print("-" * 55)
print(f"{'Scenario':<32} {'First Sustainable Year'}")
print("-" * 55)
for name, results in all_scenarios.items():
    yr_found = next((yr for yr in proj_years
                     if results[yr]['gap_percent'] < 80), None)
    print(f"{name:<32} {yr_found if yr_found else 'Never (within 2026-2030)'}")

print(f"\nCurrent charge per flight:            ₦{current_charge:,.0f}")
print(f"Inflation-required charge by 2030:    ₦{required_charge_2030:,.0f}")


In [ ]:
# ── BATCH EXPORT ALL CHARTS (300 DPI) ──
import os

export_folder = os.path.join(os.path.expanduser("~"), "Documents", "NAMA_Charts_2")
os.makedirs(export_folder, exist_ok=True)

def save(name):
    plt.savefig(os.path.join(export_folder, name), dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ {name}")

print("Exporting charts...")


# Chart 1: Monthly Revenue
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_monthly['REVENUE']/1e6, color='blue', linewidth=1.5)
ax.set_title('NAMA Yola Monthly Nominal Revenue (Jan 2021 – Sep 2025)', fontsize=13)
ax.set_xlabel('Month Index'); ax.set_ylabel('Revenue (₦ Millions)')
ax.grid(True, alpha=0.3); plt.tight_layout()
save('chart_01_monthly_revenue.png')


# Chart 2: Real Value Erosion
years_list = list(range(2008, 2026))
cum = 1.0
rv = [1.0]

for yr in range(2009, 2026):
    cum *= (1 + inflation_data[yr])
    rv.append(1/cum)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(years_list, rv, color='red', marker='o', linewidth=2)
ax.fill_between(years_list, rv, 1, alpha=0.1, color='red')
ax.set_title('Real Value of ₦1 of 2008 NAMA Yola Revenue (2008–2025)', fontsize=13)
ax.set_xlabel('Year')
ax.set_ylabel('Real Value (₦, 2008 base)')
ax.axhline(y=rv[-1], color='gray', linestyle='--', linewidth=1)
ax.annotate(
    '₦0.09 remaining\n(90.9% lost)',
    xy=(2025, rv[-1]),
    xytext=(2019, 0.25),
    arrowprops=dict(arrowstyle='->', color='black'),
    fontsize=10,
    color='red'
)
plt.xticks(years_list, rotation=45)
plt.tight_layout()
save('chart_02_real_value_erosion.png')


# Chart 3: Retrospective Gap
yrs  = annual_actual['YEAR'].astype(int).values
act  = annual_actual['ACTUAL_REVENUE'].values
adj  = annual_actual['ADJUSTED_REVENUE'].values
gap  = annual_actual['REVENUE_GAP'].values
gpct = annual_actual['GAP_PERCENT'].values

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
ax1.bar(yrs-0.2, act/1e6, width=0.4, label='Actual Revenue (Nominal)', color='steelblue')
ax1.bar(yrs+0.2, adj/1e6, width=0.4, label='Inflation-Adjusted Revenue', color='red', alpha=0.7)
ax1.set_title('NAMA Yola: Actual vs Inflation-Adjusted Revenue (2021–2025)', fontsize=13)
ax1.set_xlabel('Year')
ax1.set_ylabel('Revenue (₦ Millions)')
ax1.legend()
ax1.set_xticks(yrs)

ax2.bar(yrs, gap/1e6, color='darkred', alpha=0.8)
ax2.set_title('Annual Revenue Gap — Purchasing Power Lost (2021–2025)', fontsize=13)
ax2.set_xlabel('Year')
ax2.set_ylabel('Gap (₦ Millions)')
ax2.set_xticks(yrs)

for y, g, p in zip(yrs, gap, gpct):
    ax2.text(y, g/1e6+1, f'{p:.1f}%', ha='center', fontsize=10, color='darkred')

plt.tight_layout()
save('chart_03_retrospective_gap.png')


# Chart 4: MC Retrospective
np.random.seed(42)
mc_results_retro = {}

for year in [2021, 2022, 2023, 2024, 2025]:
    actual_rev = annual_actual.loc[
        annual_actual['YEAR'] == year,
        'ACTUAL_REVENUE'
    ].values[0]

    gaps = []
    reals = []

    for _ in range(N_ITERATIONS):
        At = 1.0

        for yr in range(2008, year):
            r = np.random.normal(inflation_data[yr], 0.05)
            At *= (1 + np.clip(r, 0.01, 0.80))

        Rr = actual_rev * At
        gaps.append(Rr - actual_rev)
        reals.append(Rr)

    mc_results_retro[year] = {
        'actual_revenue': actual_rev,
        'mean_real_revenue': np.mean(reals),
        'mean_gap': np.mean(gaps),
        'gap_5th': np.percentile(gaps, 5),
        'gap_95th': np.percentile(gaps, 95),
        'gap_percent': (np.mean(gaps) / np.mean(reals)) * 100
    }

det_gaps = {
    yr: annual_actual.loc[
        annual_actual['YEAR'] == yr,
        'REVENUE_GAP'
    ].values[0]
    for yr in [2021, 2022, 2023, 2024, 2025]
}

yrl = list(mc_results_retro.keys())
actr = [mc_results_retro[y]['actual_revenue']/1e6 for y in yrl]
mr   = [mc_results_retro[y]['mean_real_revenue']/1e6 for y in yrl]
mg   = [mc_results_retro[y]['mean_gap']/1e6 for y in yrl]
g5   = [mc_results_retro[y]['gap_5th']/1e6 for y in yrl]
g95  = [mc_results_retro[y]['gap_95th']/1e6 for y in yrl]
dv   = [det_gaps[y]/1e6 for y in yrl]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

ax1.plot(yrl, actr, 'bo-', linewidth=2, markersize=8,
         label='Actual Nominal Revenue')
ax1.plot(yrl, mr, 'ro-', linewidth=2, markersize=8,
         label='MC Mean Inflation-Required Revenue')

ax1.fill_between(
    yrl,
    [g5[i] + actr[i] for i in range(len(yrl))],
    [g95[i] + actr[i] for i in range(len(yrl))],
    alpha=0.2,
    color='red',
    label='90% CI'
)

ax1.set_title(
    'MC Retrospective: Actual vs Inflation-Required Revenue (2021–2025)',
    fontsize=12
)
ax1.set_xlabel('Year')
ax1.set_ylabel('Revenue (₦ Millions)')
ax1.legend()
ax1.set_xticks(yrl)

ax2.plot(yrl, mg, 'ro-', linewidth=2, markersize=8,
         label='MC Mean Gap')
ax2.fill_between(yrl, g5, g95, alpha=0.2, color='red', label='90% CI')
ax2.plot(yrl, g5, 'r--', linewidth=1, alpha=0.5)
ax2.plot(yrl, g95, 'r--', linewidth=1, alpha=0.5)

ax2.scatter(
    yrl, dv,
    color='darkred',
    zorder=5,
    s=80,
    marker='D',
    label='Deterministic Gap'
)

for y, g in zip(yrl, mg):
    ax2.annotate(
        f'₦{g:.1f}M',
        xy=(y, g),
        xytext=(0, 10),
        textcoords='offset points',
        ha='center',
        fontsize=9,
        color='darkred'
    )

ax2.set_title(
    'MC Revenue Gap with 90% Confidence Bands (2021–2025)',
    fontsize=12
)
ax2.set_xlabel('Year')
ax2.set_ylabel('Revenue Gap (₦ Millions)')
ax2.legend()
ax2.set_xticks(yrl)

plt.tight_layout()
save('chart_04_mc_retrospective.png')


# Chart 5: Sensitivity Analysis
np.random.seed(42)
sensitivity_results = {}

for g in g_values:
    yr_res = {}

    for year in proj_years:
        nom = []
        req = []
        gaps = []

        for _ in range(N_ITERATIONS):
            nf = year - 2025
            fi = np.clip(
                np.random.normal(
                    mean_future_inflation,
                    std_future_inflation,
                    nf
                ),
                0.01,
                0.80
            )

            yfb = year - 2025
            pm = base_movements * ((1 + cagr_movements) ** yfb)
            ac = avg_charge_per_flight * ((1 + g) ** yfb)
            nr = pm * ac

            fAt = np.prod(1 + fi)
            fA = A_base * fAt
            Rr = pm * (avg_charge_per_flight * fA)

            nom.append(nr)
            req.append(Rr)
            gaps.append(Rr - nr)

        yr_res[year] = {
            'mean_nominal': np.mean(nom),
            'mean_required': np.mean(req),
            'mean_gap': np.mean(gaps),
            'gap_percent': (np.mean(gaps) / np.mean(req)) * 100
        }

    sensitivity_results[g] = yr_res

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

colors_s = plt.cm.RdYlGn_r(
    np.linspace(0.1, 0.9, len(g_values))
)

for i, g in enumerate(g_values):
    ax1.plot(
        proj_years,
        [sensitivity_results[g][y]['gap_percent'] for y in proj_years],
        marker='o',
        color=colors_s[i],
        linewidth=2,
        label=f'g={g*100:.0f}%'
    )

ax1.set_title(
    'Revenue Gap % by Year\nUnder Different Adjustment Rates',
    fontsize=12
)
ax1.set_xlabel('Year')
ax1.set_ylabel('Gap %')
ax1.legend(loc='upper left', fontsize=9)
ax1.set_xticks(proj_years)
ax1.grid(True, alpha=0.3)

for year in proj_years:
    ax2.plot(
        [g*100 for g in g_values],
        [sensitivity_results[g][year]['gap_percent'] for g in g_values],
        marker='o',
        linewidth=2,
        label=str(year)
    )

ax2.set_title(
    'Revenue Gap % by Adjustment Rate\nFor Each Year',
    fontsize=12
)
ax2.set_xlabel('Annual Adjustment Rate (%)')
ax2.set_ylabel('Gap %')
ax2.legend(title='Year', fontsize=9)
ax2.set_xticks([g*100 for g in g_values])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
save('chart_05_sensitivity_analysis.png')


# Chart 6: Scenario Comparison
np.random.seed(42)

def run_scenario(g_val, gamma=None):
    res = {}

    for year in proj_years:
        nom = []
        req = []
        gaps = []
        gpcts = []

        for _ in range(N_ITERATIONS):
            nf = year - 2025

            fi = np.clip(
                np.random.normal(
                    mean_future_inflation,
                    std_future_inflation,
                    nf
                ),
                0.01,
                0.80
            )

            yfb = year - 2025

            if gamma:
                pm = base_movements * ((1 + gamma) ** yfb)
                nr = pm * avg_charge_per_flight
            else:
                pm = base_movements * ((1 + cagr_movements) ** yfb)
                nr = pm * (
                    avg_charge_per_flight *
                    ((1 + g_val) ** yfb)
                )

            fAt = np.prod(1 + fi)
            fA = A_base * fAt

            Rr = pm * (avg_charge_per_flight * fA)

            G = Rr - nr
            gp = G / Rr

            nom.append(nr)
            req.append(Rr)
            gaps.append(G)
            gpcts.append(gp)

        ga = np.array(gaps)
        gpa = np.array(gpcts)

        res[year] = {
            'mean_nominal': np.mean(nom),
            'mean_required': np.mean(req),
            'mean_gap': np.mean(ga),
            'gap_5th': np.percentile(ga, 5),
            'gap_95th': np.percentile(ga, 95),
            'gap_percent': np.mean(gpa) * 100,
            'prob_unsustainable': np.mean(
                gpa > SUSTAINABILITY_THRESHOLD
            ) * 100,
            'prob_sustain': np.mean(
                gpa <= SUSTAINABILITY_THRESHOLD
            ) * 100
        }

    return res


baseline_results = run_scenario(0)

scenario_results = {
    'S2_Conservative': run_scenario(0.15),
    'S3_Moderate': run_scenario(0.30),
    'S4_Aggressive': run_scenario(0.50)
}

traffic_results = run_scenario(0, gamma=0.15)

s1n = [baseline_results[y]['mean_nominal']/1e6 for y in proj_years]
s2n = [scenario_results['S2_Conservative'][y]['mean_nominal']/1e6 for y in proj_years]
s3n = [scenario_results['S3_Moderate'][y]['mean_nominal']/1e6 for y in proj_years]
s4n = [scenario_results['S4_Aggressive'][y]['mean_nominal']/1e6 for y in proj_years]
s5n = [traffic_results[y]['mean_nominal']/1e6 for y in proj_years]

req = [baseline_results[y]['mean_required']/1e6 for y in proj_years]

s1g = [baseline_results[y]['gap_percent'] for y in proj_years]
s2g = [scenario_results['S2_Conservative'][y]['gap_percent'] for y in proj_years]
s3g = [scenario_results['S3_Moderate'][y]['gap_percent'] for y in proj_years]
s4g = [scenario_results['S4_Aggressive'][y]['gap_percent'] for y in proj_years]
s5g = [traffic_results[y]['gap_percent'] for y in proj_years]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 11))

ax1.plot(
    proj_years, req, 'k--',
    linewidth=2,
    label='Inflation-Required Revenue'
)
ax1.plot(
    proj_years, s1n, 'ro-',
    linewidth=2, markersize=7,
    label='S1: Baseline'
)
ax1.plot(
    proj_years, s2n, 'yo-',
    linewidth=2, markersize=7,
    label='S2: Conservative (g=15%)'
)
ax1.plot(
    proj_years, s3n, 'bo-',
    linewidth=2, markersize=7,
    label='S3: Moderate (g=30%)'
)
ax1.plot(
    proj_years, s4n, 'go-',
    linewidth=2, markersize=7,
    label='S4: Aggressive (g=50%)'
)
ax1.plot(
    proj_years, s5n, 'mo-',
    linewidth=2, markersize=7,
    label='S5: Traffic Growth (γ=15%)'
)

ax1.set_title(
    'Projected Annual Revenue Under Five Scenarios (2026–2030)',
    fontsize=13
)
ax1.set_xlabel('Year')
ax1.set_ylabel('Revenue (₦ Millions)')
ax1.legend(loc='upper left', fontsize=9)
ax1.set_xticks(proj_years)
ax1.grid(True, alpha=0.3)

ax2.plot(
    proj_years, s1g, 'ro-',
    linewidth=2, markersize=7,
    label='S1: Baseline'
)
ax2.plot(
    proj_years, s2g, 'yo-',
    linewidth=2, markersize=7,
    label='S2: Conservative (g=15%)'
)
ax2.plot(
    proj_years, s3g, 'bo-',
    linewidth=2, markersize=7,
    label='S3: Moderate (g=30%)'
)
ax2.plot(
    proj_years, s4g, 'go-',
    linewidth=2, markersize=7,
    label='S4: Aggressive (g=50%)'
)
ax2.plot(
    proj_years, s5g, 'mo-',
    linewidth=2, markersize=7,
    label='S5: Traffic Growth (γ=15%)'
)

ax2.axhline(
    y=80,
    color='black',
    linestyle='--',
    linewidth=1.5,
    label='80% Threshold'
)

ax2.fill_between(
    proj_years, 80, 105,
    alpha=0.04,
    color='red'
)

ax2.fill_between(
    proj_years, 50, 80,
    alpha=0.04,
    color='green'
)

ax2.text(
    2029.2, 97,
    'Unsustainable Zone',
    fontsize=9,
    color='red',
    alpha=0.8,
    ha='center'
)

ax2.text(
    2029.2, 55,
    'Sustainable Zone',
    fontsize=9,
    color='green',
    alpha=0.8,
    ha='center'
)

ax2.set_title(
    'Revenue Gap % Under Five Scenarios (2026–2030)',
    fontsize=13
)
ax2.set_xlabel('Year')
ax2.set_ylabel('Gap %')
ax2.legend(loc='upper left', fontsize=9)
ax2.set_xticks(proj_years)
ax2.set_ylim(50, 105)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
save('chart_06_scenario_comparison.png')


# Charts 7 & 8: Fan Charts
np.random.seed(42)

colors = {
    'S1_Baseline': 'red',
    'S2_Conservative': 'goldenrod',
    'S3_Moderate': 'blue',
    'S4_Aggressive': 'green',
    'S5_Traffic': 'purple'
}

labels = {
    'S1_Baseline': 'S1: Baseline (Frozen)',
    'S2_Conservative': 'S2: Conservative (g=15%)',
    'S3_Moderate': 'S3: Moderate (g=30%)',
    'S4_Aggressive': 'S4: Aggressive (g=50%)',
    'S5_Traffic': 'S5: Traffic Growth (γ=15%)'
}

sc_map = {
    'S1_Baseline': (0, None),
    'S2_Conservative': (0.15, None),
    'S3_Moderate': (0.30, None),
    'S4_Aggressive': (0.50, None),
    'S5_Traffic': (0, 0.15)
}

fan_results = {}

for sn, (g, gm) in sc_map.items():
    ydata = {}

    for year in proj_years:
        arr = []

        for _ in range(N_ITERATIONS):
            nf = year - 2025

            fi = np.clip(
                np.random.normal(
                    mean_future_inflation,
                    std_future_inflation,
                    nf
                ),
                0.01,
                0.80
            )

            yfb = year - 2025

            if gm:
                pm = base_movements * ((1 + gm) ** yfb)
                nr = pm * avg_charge_per_flight
            else:
                pm = base_movements * ((1 + cagr_movements) ** yfb)
                nr = pm * (
                    avg_charge_per_flight *
                    ((1 + g) ** yfb)
                )

            fAt = np.prod(1 + fi)
            fA = A_base * fAt

            Rr = pm * (avg_charge_per_flight * fA)

            arr.append((Rr - nr) / Rr * 100)

        a = np.array(arr)

        ydata[year] = {
            'mean': np.mean(a),
            'p5': np.percentile(a, 5),
            'p10': np.percentile(a, 10),
            'p25': np.percentile(a, 25),
            'p50': np.percentile(a, 50),
            'p75': np.percentile(a, 75),
            'p90': np.percentile(a, 90),
            'p95': np.percentile(a, 95)
        }

    fan_results[sn] = ydata
    print(f"Fan {sn} ✅")


# Chart 7: Combined
fig, ax = plt.subplots(figsize=(13, 8))

for sn, yd in fan_results.items():
    c = colors[sn]

    ax.plot(
        proj_years,
        [yd[y]['mean'] for y in proj_years],
        color=c,
        linewidth=2,
        marker='o',
        markersize=6,
        label=labels[sn]
    )

    ax.fill_between(
        proj_years,
        [yd[y]['p5'] for y in proj_years],
        [yd[y]['p95'] for y in proj_years],
        alpha=0.12,
        color=c
    )

ax.axhline(
    y=80,
    color='black',
    linestyle='--',
    linewidth=1.5,
    label='80% Threshold'
)

ax.fill_between(
    proj_years, 80, 105,
    alpha=0.04,
    color='red'
)

ax.fill_between(
    proj_years, 50, 80,
    alpha=0.04,
    color='green'
)

ax.text(
    2029.2, 97,
    'Unsustainable\nZone',
    fontsize=9,
    color='red',
    alpha=0.8,
    ha='center'
)

ax.text(
    2029.2, 55,
    'Sustainable\nZone',
    fontsize=9,
    color='green',
    alpha=0.8,
    ha='center'
)

ax.set_title(
    'Fan Chart: Revenue Gap % Under Five Scenarios (2026–2030)\n'
    'Shaded = 90% CI',
    fontsize=13
)

ax.set_xlabel('Year')
ax.set_ylabel('Gap %')
ax.legend(loc='upper left', fontsize=9)
ax.set_xticks(proj_years)
ax.set_ylim(50, 105)
ax.grid(True, alpha=0.3)

plt.tight_layout()
save('chart_07_combined_fan_chart.png')


# Chart 8: Individual
fig, axes = plt.subplots(2, 3, figsize=(16, 11))
axes = axes.flatten()

for idx, (sn, yd) in enumerate(fan_results.items()):
    ax = axes[idx]
    c = colors[sn]

    ax.fill_between(
        proj_years,
        [yd[y]['p5'] for y in proj_years],
        [yd[y]['p95'] for y in proj_years],
        alpha=0.15,
        color=c,
        label='90% CI'
    )

    ax.fill_between(
        proj_years,
        [yd[y]['p10'] for y in proj_years],
        [yd[y]['p90'] for y in proj_years],
        alpha=0.20,
        color=c,
        label='80% CI'
    )

    ax.fill_between(
        proj_years,
        [yd[y]['p25'] for y in proj_years],
        [yd[y]['p75'] for y in proj_years],
        alpha=0.35,
        color=c,
        label='50% CI'
    )

    means = [yd[y]['mean'] for y in proj_years]

    ax.plot(
        proj_years,
        means,
        color=c,
        linewidth=2.5,
        marker='o',
        markersize=7,
        label='Mean',
        zorder=5
    )

    ax.axhline(
        y=80,
        color='black',
        linestyle='--',
        linewidth=1.2,
        alpha=0.8
    )

    ax.fill_between(
        proj_years, 80, 105,
        alpha=0.04,
        color='red'
    )

    ax.fill_between(
        proj_years, 40, 80,
        alpha=0.04,
        color='green'
    )

    ax.text(
        2026.1, 81.5,
        '80% threshold',
        fontsize=7.5,
        color='black',
        alpha=0.8
    )

    ax.annotate(
        f"{means[-1]:.1f}%",
        xy=(2030, means[-1]),
        xytext=(5, 0),
        textcoords='offset points',
        fontsize=8,
        color=c,
        fontweight='bold'
    )

    ax.set_title(
        labels[sn],
        fontsize=10,
        fontweight='bold',
        color=c
    )

    ax.set_xlabel('Year', fontsize=9)
    ax.set_ylabel('Gap %', fontsize=9)
    ax.set_xticks(proj_years)
    ax.set_ylim(40, 105)
    ax.legend(fontsize=7.5, loc='lower left')
    ax.grid(True, alpha=0.3)

axes[5].set_visible(False)

fig.suptitle(
    'Individual Fan Charts: Revenue Gap % by Scenario (2026–2030)\n'
    '50%, 80%, and 90% CIs',
    fontsize=13,
    fontweight='bold',
    y=1.01
)

plt.tight_layout()
save('chart_08_individual_fan_charts.png')


# ── EXPORT COMPLETE ──
print(f"\n✅ All 8 charts exported to:")
print(export_folder)


In [ ]:
# ── AUDIT: Implied TNC and ENC rates per scenario ──
print("IMPLIED TNC AND ENC RATES UNDER EACH SCENARIO")
print("=" * 80)

r_T = 6000   # current TNC unit rate
r_E = 2000   # current ENC unit rate
d   = 5.5    # distance factor

scenarios = {
    'S1: Baseline (g=0%)':      0.00,
    'S2: Conservative (g=15%)': 0.15,
    'S3: Moderate (g=30%)':     0.30,
    'S4: Aggressive (g=50%)':   0.50,
}

years_proj = [2026, 2027, 2028, 2029, 2030]

for name, g in scenarios.items():
    print(f"\n{name}")
    print(f"{'Year':<8} {'r*_T (₦)':<16} {'r*_E (₦)':<16} {'Charge/flight (w=1.0)':<25} {'Charge/flight (w=1.1)'}")
    print("-" * 80)
    for year in years_proj:
        t = year - 2025
        rT = r_T * ((1 + g) ** t)
        rE = r_E * ((1 + g) ** t)
        # Charge per flight at w=1.0 (minimum)
        charge_w1  = (1.0 * rT) + (1.0 * d * rE)
        # Charge per flight at w=1.1 (typical MAX AIR)
        charge_w11 = (1.1 * rT) + (1.1 * d * rE)
        print(f"{year:<8} ₦{rT:<14,.0f} ₦{rE:<14,.0f} ₦{charge_w1:<23,.0f} ₦{charge_w11:,.0f}")

# Full inflation indexing benchmark
print(f"\nFULL INFLATION INDEXING BENCHMARK (what rates SHOULD be by 2030)")
print("-" * 50)
full_factor = cumulative_factor_lagged(2026) * ((1 + mean_future_inflation) ** 5)
rT_required = r_T * full_factor
rE_required = r_E * full_factor
charge_required = (1.0 * rT_required) + (1.0 * d * rE_required)
print(f"Required TNC by 2030:          ₦{rT_required:,.0f}")
print(f"Required ENC by 2030:          ₦{rE_required:,.0f}")
print(f"Required charge/flight (w=1):  ₦{charge_required:,.0f}")
